In [1]:
from pathlib import Path
import torch
from datasets import load_from_disk
from transformers import AutoTokenizer

# Project paths
PROJECT_DIR = Path(r"E:\punctuation_project")
DATA_DIR = PROJECT_DIR / "data" / "processed"
MODEL_DIR = PROJECT_DIR / "models" / "hindi_english_translation"

MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# Load existing processed dataset
translation_processed = load_from_disk(
    str(DATA_DIR / "translation_processed")
)

# Load the tokenizer used during dataset preprocessing
mt5_tokenizer = AutoTokenizer.from_pretrained(
    "google/mt5-small"
)

print("\nDataset:")
print(translation_processed)

print("\nModel save location:", MODEL_DIR)

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU

Dataset:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1320349
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 165044
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 165044
    })
})

Model save location: E:\punctuation_project\models\hindi_english_translation


In [2]:
sample = translation_processed["train"][0]

print("Dataset columns:", translation_processed["train"].column_names)

print("\nHindi input:")
print(mt5_tokenizer.decode(
    sample["input_ids"],
    skip_special_tokens=True
))

print("\nEnglish target:")
print(mt5_tokenizer.decode(
    sample["labels"],
    skip_special_tokens=True
))

print("\nToken lengths:")
print("Hindi:", len(sample["input_ids"]))
print("English:", len(sample["labels"]))

print("\nTokenizer vocabulary size:", len(mt5_tokenizer))

Dataset columns: ['input_ids', 'attention_mask', 'labels']

Hindi input:
translate Hindi to English: एक दुसरे से करते हैं प्यार हम

English target:
Ek Doosre Se Karte Hain Pyaar Hum

Token lengths:
Hindi: 18
English: 11

Tokenizer vocabulary size: 250100


In [3]:
for i in range(10):
    sample = translation_processed["train"][i]

    hindi = mt5_tokenizer.decode(
        sample["input_ids"],
        skip_special_tokens=True
    )

    target = mt5_tokenizer.decode(
        sample["labels"],
        skip_special_tokens=True
    )

    print(f"\nExample {i + 1}")
    print("Hindi :", hindi)
    print("Target:", target)


Example 1
Hindi : translate Hindi to English: एक दुसरे से करते हैं प्यार हम
Target: Ek Doosre Se Karte Hain Pyaar Hum

Example 2
Hindi : translate Hindi to English: राममोहन राय के धार्मिक अभियान की पृष्ठभूमि का विवेचन करते हुए अभिताभ मुखर्जी ने लगातार यह विचार प्रकट किया है किः राममोहन राय मूलतः उस अर्थ में एक धर्म-सुधारक नहीं थे, जिस अर्थ में मध्यकालीन भारतीय इतिहास के रामानन्द, कबीर, नानक और चैतन्य धर्म-सुधारक थे।
Target: Discussing the background of Rammohun 's religious crusade, Amitabha Mukherjee has pertinently observed:' Rammohun was not essentially a religious reformer in the sense that Ramananda, Kabir, Nanak and Chaitanya were during the medieval period of India history.

Example 3
Hindi : translate Hindi to English: मौक़ापरस्ती
Target: opportunism

Example 4
Hindi : translate Hindi to English: सिंधुजा
Target: shell

Example 5
Hindi : translate Hindi to English: इससे पत्तियों में रंग आ जाता है।
Target: So the colour is brought out.

Example 6
Hindi : translate Hindi to Engli

In [4]:
import importlib.util

for package in ["langid", "langdetect", "fasttext"]:
    print(package, ":", importlib.util.find_spec(package))

langid : None
langdetect : None
fasttext : None


In [5]:
%pip install langid

     ---------------------------------------- 0.0/1.9 MB ? eta -:--:--
     ----- ---------------------------------- 0.3/1.9 MB ? eta -:--:--
     ---------------------------------------- 1.9/1.9 MB 8.9 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for langid: filename=langid-1.1.6-py3-none-any.whl size=1941227 sha256=6b8e1e220d2f583cccc7b1188d5ecc14432062c8ff1bc56443d1f8d51637fdcd
  Stored in directory: c:\users\home\appdata\local\pip\cache\wheels\32\6a\b6\b7eb43a6ad55b139c15c5daa29f3707659cfa6944d3c696f5b
Successfully built langid
Note: you may need to restart the kernel to use updated packages.


In [6]:
import langid
import random
from collections import Counter

random.seed(42)

train_data = translation_processed["train"]

sample_indices = random.sample(
    range(len(train_data)),
    1000
)

language_counts = Counter()
suspicious_examples = []

for idx in sample_indices:
    sample = train_data[idx]

    target = mt5_tokenizer.decode(
        sample["labels"],
        skip_special_tokens=True
    ).strip()

    if not target:
        continue

    language, confidence = langid.classify(target)
    language_counts[language] += 1

    if language != "en":
        suspicious_examples.append(
            (language, target)
        )

print("Detected target languages in 1,000 samples:")
for language, count in language_counts.most_common():
    print(f"{language}: {count}")

print("\nExamples not classified as English:")
for language, text in suspicious_examples[:20]:
    print(f"[{language}] {text}")

Detected target languages in 1,000 samples:
en: 909
de: 21
es: 18
fr: 11
lt: 8
it: 4
et: 4
mt: 4
nl: 3
pl: 2
pt: 2
da: 2
sl: 2
sw: 1
hi: 1
sv: 1
hu: 1
ca: 1
lv: 1
id: 1
hr: 1
vi: 1
sk: 1

Examples not classified as English:
[de] shaft
[lt] Proprietary
[it] Iliopsoas
[sw] Maha Sakthi india (Google Book - Writer - Vedhpradhap vaidhik)
[et] ab initio
[de] Maximum distance:
[fr] poison
[de] Trumpet newsreader
[fr] File processing
[es] Undo Move% d item
[lt] regular - ir verb
[es] Malayo - Polynesian languages
[hi] “Oh, my God! ” “हे
[de] being
[mt] lilliputian
[es] Turks And Caicos Islands
[et] dusty
[mt] GTKmm (Simple)
[pl] Lrzip (. lrz)
[et] Move Tab _ Right


In [7]:
import numpy as np

train_data = translation_processed["train"]

indices = np.random.default_rng(42).choice(
    len(train_data),
    size=10000,
    replace=False
)

source_lengths = []
target_lengths = []

for idx in indices:
    sample = train_data[int(idx)]

    source_lengths.append(
        len(sample["input_ids"])
    )

    target_lengths.append(
        len(sample["labels"])
    )

for name, lengths in [
    ("Hindi source", source_lengths),
    ("English target", target_lengths)
]:
    print(f"\n{name}")
    print("Mean:", round(np.mean(lengths), 2))
    print("Median:", int(np.percentile(lengths, 50)))
    print("90th percentile:", int(np.percentile(lengths, 90)))
    print("95th percentile:", int(np.percentile(lengths, 95)))
    print("99th percentile:", int(np.percentile(lengths, 99)))
    print("Maximum:", max(lengths))


Hindi source
Mean: 34.82
Median: 25
90th percentile: 74
95th percentile: 99
99th percentile: 128
Maximum: 128

English target
Mean: 20.73
Median: 14
90th percentile: 47
95th percentile: 61
99th percentile: 104
Maximum: 128


In [8]:
print("Vocabulary size:", len(mt5_tokenizer))

print("PAD token:", mt5_tokenizer.pad_token)
print("PAD ID:", mt5_tokenizer.pad_token_id)

print("EOS token:", mt5_tokenizer.eos_token)
print("EOS ID:", mt5_tokenizer.eos_token_id)

print("UNK token:", mt5_tokenizer.unk_token)
print("UNK ID:", mt5_tokenizer.unk_token_id)

print("Decoder start token:", mt5_tokenizer.pad_token_id)

Vocabulary size: 250100
PAD token: <pad>
PAD ID: 0
EOS token: </s>
EOS ID: 1
UNK token: <unk>
UNK ID: 2
Decoder start token: 0


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Model configuration
VOCAB_SIZE = len(mt5_tokenizer)
D_MODEL = 128
NUM_HEADS = 4
NUM_ENCODER_LAYERS = 2
NUM_DECODER_LAYERS = 2
FFN_DIM = 256
DROPOUT = 0.1

MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 64

PAD_ID = mt5_tokenizer.pad_token_id
EOS_ID = mt5_tokenizer.eos_token_id
START_ID = mt5_tokenizer.pad_token_id


class HindiEnglishTransformer(nn.Module):

    def __init__(self):
        super().__init__()

        # One shared embedding matrix for Hindi and English
        self.embedding = nn.Embedding(
            VOCAB_SIZE,
            D_MODEL,
            padding_idx=PAD_ID
        )

        # Positional embeddings
        self.source_position = nn.Embedding(
            MAX_SOURCE_LENGTH, D_MODEL
        )

        self.target_position = nn.Embedding(
            MAX_TARGET_LENGTH, D_MODEL
        )

        # Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=NUM_HEADS,
            dim_feedforward=FFN_DIM,
            dropout=DROPOUT,
            batch_first=True,
            activation="gelu"
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=NUM_ENCODER_LAYERS
        )

        # Decoder
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=D_MODEL,
            nhead=NUM_HEADS,
            dim_feedforward=FFN_DIM,
            dropout=DROPOUT,
            batch_first=True,
            activation="gelu"
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=NUM_DECODER_LAYERS
        )

        self.dropout = nn.Dropout(DROPOUT)

        # Output projection shares weights with the embedding matrix.
        # This avoids creating another 250,100 x 128 parameter matrix.

    def forward(self, source_ids, decoder_ids):

        batch_size, source_length = source_ids.shape
        target_length = decoder_ids.shape[1]

        source_positions = torch.arange(
            source_length, device=source_ids.device
        ).unsqueeze(0)

        target_positions = torch.arange(
            target_length, device=decoder_ids.device
        ).unsqueeze(0)

        # Embed Hindi input
        source = self.embedding(source_ids)
        source = source + self.source_position(source_positions)
        source = self.dropout(source)

        source_padding_mask = source_ids.eq(PAD_ID)

        # Encode Hindi
        memory = self.encoder(
            source,
            src_key_padding_mask=source_padding_mask
        )

        # Embed English decoder input
        target = self.embedding(decoder_ids)
        target = target + self.target_position(target_positions)
        target = self.dropout(target)

        target_padding_mask = decoder_ids.eq(PAD_ID)

        # Prevent decoder from looking at future tokens
        causal_mask = torch.triu(
            torch.ones(
                target_length,
                target_length,
                device=decoder_ids.device,
                dtype=torch.bool
            ),
            diagonal=1
        )

        # Decode English
        decoded = self.decoder(
            tgt=target,
            memory=memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=target_padding_mask,
            memory_key_padding_mask=source_padding_mask
        )

        # Project to vocabulary using the shared embedding weights
        logits = F.linear(decoded, self.embedding.weight)

        return logits


# Create model
model = HindiEnglishTransformer().to(device)

total_parameters = sum(
    parameter.numel() for parameter in model.parameters()
)

print("Model created successfully.")
print("Total parameters:", f"{total_parameters:,}")
print("Total parameter size:", round(
    total_parameters * 4 / (1024 ** 2), 2
), "MB (FP32)")

Model created successfully.
Total parameters: 32,699,904
Total parameter size: 124.74 MB (FP32)


In [10]:
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

dummy_source = torch.randint(
    3, VOCAB_SIZE, (1, MAX_SOURCE_LENGTH), device=device
)

dummy_target = torch.randint(
    3, VOCAB_SIZE, (1, MAX_TARGET_LENGTH), device=device
)

with torch.no_grad():
    dummy_logits = model(dummy_source, dummy_target)

print("Output shape:", tuple(dummy_logits.shape))

print(
    "Peak GPU memory:",
    round(torch.cuda.max_memory_allocated() / (1024 ** 3), 3),
    "GB"
)

del dummy_source, dummy_target, dummy_logits
torch.cuda.empty_cache()

Output shape: (1, 64, 250100)
Peak GPU memory: 0.19 GB


In [11]:
sample = translation_processed["train"][0]

target_ids = sample["labels"]

print("First 30 target token IDs:")
print(target_ids[:30])

print("\nLast 20 target token IDs:")
print(target_ids[-20:])

print("\nNumber of padding tokens:", target_ids.count(PAD_ID))
print("Contains EOS:", EOS_ID in target_ids)

print("\nDecoded target:")
print(
    mt5_tokenizer.decode(
        target_ids,
        skip_special_tokens=True
    )
)

First 30 target token IDs:
[5398, 1272, 337, 380, 746, 63324, 70545, 363, 121839, 27969, 1]

Last 20 target token IDs:
[5398, 1272, 337, 380, 746, 63324, 70545, 363, 121839, 27969, 1]

Number of padding tokens: 0
Contains EOS: True

Decoded target:
Ek Doosre Se Karte Hain Pyaar Hum


In [12]:
from collections import Counter
import numpy as np
import json

COMPACT_VOCAB_SIZE = 32000
VOCAB_SAMPLE_SIZE = 100000

rng = np.random.default_rng(42)

vocab_indices = rng.choice(
    len(translation_processed["train"]),
    size=VOCAB_SAMPLE_SIZE,
    replace=False
)

token_counts = Counter()

for idx in vocab_indices:
    sample = translation_processed["train"][int(idx)]

    token_counts.update(sample["input_ids"])
    token_counts.update(sample["labels"])

# Always retain special tokens
special_ids = [
    PAD_ID,
    EOS_ID,
    mt5_tokenizer.unk_token_id
]

frequent_ids = [
    token_id
    for token_id, count in token_counts.most_common()
    if token_id not in special_ids
]

selected_token_ids = (
    special_ids + frequent_ids[:COMPACT_VOCAB_SIZE - len(special_ids)]
)

# Original token ID -> compact token ID
original_to_compact = {
    original_id: compact_id
    for compact_id, original_id in enumerate(selected_token_ids)
}

# Compact token ID -> original token ID
compact_to_original = selected_token_ids

# Save vocabulary mapping
with open(MODEL_DIR / "vocabulary_mapping.json", "w") as f:
    json.dump(
        [int(x) for x in compact_to_original],
        f
    )

print("Compact vocabulary size:", len(selected_token_ids))
print("Original vocabulary size:", len(mt5_tokenizer))
print("Unknown-token ID:", original_to_compact[mt5_tokenizer.unk_token_id])
print("Vocabulary mapping saved.")

Compact vocabulary size: 32000
Original vocabulary size: 250100
Unknown-token ID: 2
Vocabulary mapping saved.


In [13]:
import gc
import numpy as np
from datasets import DatasetDict

# Select reproducible subsets
rng = np.random.default_rng(42)

train_indices = np.sort(
    rng.choice(
        len(translation_processed["train"]),
        size=100000,
        replace=False
    )
)

val_indices = np.sort(
    rng.choice(
        len(translation_processed["validation"]),
        size=5000,
        replace=False
    )
)

train_subset = translation_processed["train"].select(
    train_indices.tolist()
)

val_subset = translation_processed["validation"].select(
    val_indices.tolist()
)

# Convert original token IDs to compact token IDs
def convert_to_compact(batch):

    new_sources = []
    new_targets = []

    for source, target in zip(
        batch["input_ids"],
        batch["labels"]
    ):

        # Convert Hindi source
        source = [
            original_to_compact.get(token, 2)
            for token in source[:MAX_SOURCE_LENGTH]
        ]

        # Preserve/add EOS
        if not source or source[-1] != EOS_ID:
            source = source[:MAX_SOURCE_LENGTH - 1] + [
                original_to_compact[EOS_ID]
            ]

        # Convert English target
        target = [
            original_to_compact.get(token, 2)
            for token in target[:MAX_TARGET_LENGTH]
        ]

        # Preserve/add EOS
        if not target or target[-1] != EOS_ID:
            target = target[:MAX_TARGET_LENGTH - 1] + [
                original_to_compact[EOS_ID]
            ]

        new_sources.append(source)
        new_targets.append(target)

    return {
        "source_ids": new_sources,
        "target_ids": new_targets
    }


compact_data = DatasetDict({
    "train": train_subset.map(
        convert_to_compact,
        batched=True,
        remove_columns=train_subset.column_names,
        desc="Converting training data"
    ),
    "validation": val_subset.map(
        convert_to_compact,
        batched=True,
        remove_columns=val_subset.column_names,
        desc="Converting validation data"
    )
})

# Save separately
compact_data.save_to_disk(
    str(DATA_DIR / "translation_compact_32k")
)

print(compact_data)

print("\nExample Hindi:")
print(
    mt5_tokenizer.decode(
        [
            compact_to_original[x]
            for x in compact_data["train"][0]["source_ids"]
        ],
        skip_special_tokens=True
    )
)

print("\nExample English:")
print(
    mt5_tokenizer.decode(
        [
            compact_to_original[x]
            for x in compact_data["train"][0]["target_ids"]
        ],
        skip_special_tokens=True
    )
)

Converting training data:   0%|          | 0/100000 [00:00<?, ? examples/s]

Converting validation data:   0%|          | 0/5000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['source_ids', 'target_ids'],
        num_rows: 100000
    })
    validation: Dataset({
        features: ['source_ids', 'target_ids'],
        num_rows: 5000
    })
})

Example Hindi:
translate Hindi to English: राममोहन राय के धार्मिक अभियान की पृष्ठभूमि का विवेचन करते हुए अभिताभ मुखर्जी ने लगातार यह विचार प्रकट किया है किः राममोहन राय मूलतः उस अर्थ में एक धर्म-सुधारक नहीं थे, जिस अर्थ में मध्यकालीन भारतीय इतिहास के रामानन्द, कबीर, नानक और चैतन्य धर्म-सुधारक थे।

Example English:
Discussing the background of Rammohun 's religious crusade, Amitabha Mukherjee hasently observed:' Rammohun was not essentially a religious reformer in the sense that Ramananda, Kabir, Nanak and Chaitanya were during the


In [14]:
total_source_tokens = 0
total_target_tokens = 0

unknown_source_tokens = 0
unknown_target_tokens = 0

targets_over_64 = 0
sources_over_128 = 0

for sample in train_subset:

    source = sample["input_ids"]
    target = sample["labels"]

    total_source_tokens += len(source)
    total_target_tokens += len(target)

    unknown_source_tokens += sum(
        token not in original_to_compact
        for token in source
    )

    unknown_target_tokens += sum(
        token not in original_to_compact
        for token in target
    )

    if len(target) > MAX_TARGET_LENGTH:
        targets_over_64 += 1

    if len(source) > MAX_SOURCE_LENGTH:
        sources_over_128 += 1


print("VOCABULARY LOSS")
print(
    "Hindi unknown-token rate:",
    round(100 * unknown_source_tokens / total_source_tokens, 2),
    "%"
)

print(
    "English unknown-token rate:",
    round(100 * unknown_target_tokens / total_target_tokens, 2),
    "%"
)

print("\nTRUNCATION")
print("English targets longer than 64:", targets_over_64)
print("Hindi sources longer than 128:", sources_over_128)

print(
    "Percentage of long English targets:",
    round(100 * targets_over_64 / len(train_subset), 2),
    "%"
)

VOCABULARY LOSS
Hindi unknown-token rate: 0.03 %
English unknown-token rate: 0.42 %

TRUNCATION
English targets longer than 64: 4371
Hindi sources longer than 128: 0
Percentage of long English targets: 4.37 %


In [15]:
MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128

def convert_to_compact(batch):

    new_sources = []
    new_targets = []

    for source, target in zip(
        batch["input_ids"],
        batch["labels"]
    ):

        source = [
            original_to_compact.get(token, 2)
            for token in source[:MAX_SOURCE_LENGTH]
        ]

        target = [
            original_to_compact.get(token, 2)
            for token in target[:MAX_TARGET_LENGTH]
        ]

        # Ensure EOS is present
        if not source or source[-1] != EOS_ID:
            source = source[:MAX_SOURCE_LENGTH - 1] + [
                original_to_compact[EOS_ID]
            ]

        if not target or target[-1] != EOS_ID:
            target = target[:MAX_TARGET_LENGTH - 1] + [
                original_to_compact[EOS_ID]
            ]

        new_sources.append(source)
        new_targets.append(target)

    return {
        "source_ids": new_sources,
        "target_ids": new_targets
    }


compact_data = DatasetDict({
    "train": train_subset.map(
        convert_to_compact,
        batched=True,
        remove_columns=train_subset.column_names,
        desc="Preparing training data"
    ),
    "validation": val_subset.map(
        convert_to_compact,
        batched=True,
        remove_columns=val_subset.column_names,
        desc="Preparing validation data"
    )
})

# Save to a new directory
compact_data.save_to_disk(
    str(DATA_DIR / "translation_compact_32k_len128")
)

print(compact_data)
print("\nMaximum target length:",
      max(len(x) for x in compact_data["train"]["target_ids"]))

print("Maximum source length:",
      max(len(x) for x in compact_data["train"]["source_ids"]))

Preparing training data:   0%|          | 0/100000 [00:00<?, ? examples/s]

Preparing validation data:   0%|          | 0/5000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/100000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['source_ids', 'target_ids'],
        num_rows: 100000
    })
    validation: Dataset({
        features: ['source_ids', 'target_ids'],
        num_rows: 5000
    })
})

Maximum target length: 128
Maximum source length: 128


In [21]:
# Updated model configuration
VOCAB_SIZE = len(compact_to_original)

D_MODEL = 128
NUM_HEADS = 4
NUM_ENCODER_LAYERS = 2
NUM_DECODER_LAYERS = 2
FFN_DIM = 256
DROPOUT = 0.1

MAX_SOURCE_LENGTH = 128
MAX_TARGET_LENGTH = 128

PAD_ID = original_to_compact[mt5_tokenizer.pad_token_id]
EOS_ID = original_to_compact[mt5_tokenizer.eos_token_id]
UNK_ID = original_to_compact[mt5_tokenizer.unk_token_id]
START_ID = PAD_ID


class HindiEnglishTransformer(nn.Module):

    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding(
            VOCAB_SIZE,
            D_MODEL,
            padding_idx=PAD_ID
        )

        self.source_position = nn.Embedding(
            MAX_SOURCE_LENGTH, D_MODEL
        )

        self.target_position = nn.Embedding(
            MAX_TARGET_LENGTH, D_MODEL
        )

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=NUM_HEADS,
            dim_feedforward=FFN_DIM,
            dropout=DROPOUT,
            batch_first=True,
            activation="gelu"
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=NUM_ENCODER_LAYERS
        )

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=D_MODEL,
            nhead=NUM_HEADS,
            dim_feedforward=FFN_DIM,
            dropout=DROPOUT,
            batch_first=True,
            activation="gelu"
        )

        self.decoder = nn.TransformerDecoder(
            decoder_layer,
            num_layers=NUM_DECODER_LAYERS
        )

        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, source_ids, decoder_ids):

        source_length = source_ids.size(1)
        target_length = decoder_ids.size(1)

        source_positions = torch.arange(
            source_length, device=source_ids.device
        ).unsqueeze(0)

        target_positions = torch.arange(
            target_length, device=decoder_ids.device
        ).unsqueeze(0)

        source = self.embedding(source_ids)
        source = source + self.source_position(source_positions)
        source = self.dropout(source)

        source_padding_mask = source_ids.eq(PAD_ID)

        memory = self.encoder(
            source,
            src_key_padding_mask=source_padding_mask
        )

        target = self.embedding(decoder_ids)
        target = target + self.target_position(target_positions)
        target = self.dropout(target)

        target_padding_mask = decoder_ids.eq(PAD_ID).clone()

        # The first decoder token is the start token, not padding.
        target_padding_mask[:, 0] = False

        causal_mask = torch.triu(
            torch.ones(
                target_length,
                target_length,
                dtype=torch.bool,
                device=decoder_ids.device
            ),
            diagonal=1
        )

        decoded = self.decoder(
            tgt=target,
            memory=memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=target_padding_mask,
            memory_key_padding_mask=source_padding_mask
        )

        logits = F.linear(decoded, self.embedding.weight)

        return logits


model = HindiEnglishTransformer().to(device)

print("Model parameters:",
      f"{sum(p.numel() for p in model.parameters()):,}")
print("Vocabulary size:", VOCAB_SIZE)
print("Maximum sequence length:", MAX_TARGET_LENGTH)

Model parameters: 4,791,296
Vocabulary size: 32000
Maximum sequence length: 128


In [17]:
from torch.utils.data import DataLoader
import torch.nn.functional as F

# Dynamically pad sequences within each batch
def collate_translation(batch):

    sources = [x["source_ids"] for x in batch]
    targets = [x["target_ids"] for x in batch]

    max_src = max(len(x) for x in sources)
    max_tgt = max(len(x) for x in targets)

    source_ids = torch.full(
        (len(batch), max_src),
        PAD_ID,
        dtype=torch.long
    )

    target_ids = torch.full(
        (len(batch), max_tgt),
        -100,
        dtype=torch.long
    )

    decoder_ids = torch.full(
        (len(batch), max_tgt),
        PAD_ID,
        dtype=torch.long
    )

    for i, (source, target) in enumerate(zip(sources, targets)):

        source_ids[i, :len(source)] = torch.tensor(source)

        # Original English target, including EOS
        target_ids[i, :len(target)] = torch.tensor(target)

        # Decoder receives START + previous target tokens
        decoder_sequence = [START_ID] + target[:-1]

        decoder_ids[i, :len(decoder_sequence)] = torch.tensor(
            decoder_sequence
        )

    return {
        "source_ids": source_ids,
        "decoder_ids": decoder_ids,
        "target_ids": target_ids
    }


BATCH_SIZE = 4

train_loader = DataLoader(
    compact_data["train"],
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_translation,
    num_workers=0,
    pin_memory=False
)

# Fetch one actual batch
batch = next(iter(train_loader))

source_ids = batch["source_ids"].to(device)
decoder_ids = batch["decoder_ids"].to(device)
target_ids = batch["target_ids"].to(device)

print("Source batch shape:", tuple(source_ids.shape))
print("Decoder batch shape:", tuple(decoder_ids.shape))
print("Target batch shape:", tuple(target_ids.shape))

# Test forward and backward
model.train()
model.zero_grad(set_to_none=True)

optimizer_test = torch.optim.AdamW(
    model.parameters(),
    lr=0.0003
)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

logits = model(source_ids, decoder_ids)

loss = F.cross_entropy(
    logits.reshape(-1, VOCAB_SIZE),
    target_ids.reshape(-1),
    ignore_index=-100
)

loss.backward()

print("\nInitial training loss:", round(loss.item(), 4))

print(
    "Peak GPU memory:",
    round(
        torch.cuda.max_memory_allocated() / (1024 ** 3),
        3
    ),
    "GB"
)

print("Backward pass successful.")

Source batch shape: (4, 128)
Decoder batch shape: (4, 66)
Target batch shape: (4, 66)

Initial training loss: 71.5867
Peak GPU memory: 0.166 GB
Backward pass successful.


In [18]:
# Reinitialize shared token embeddings
nn.init.normal_(
    model.embedding.weight,
    mean=0.0,
    std=0.02
)

# Reinitialize positional embeddings
nn.init.normal_(
    model.source_position.weight,
    mean=0.0,
    std=0.02
)

nn.init.normal_(
    model.target_position.weight,
    mean=0.0,
    std=0.02
)

# Keep padding embedding at zero
with torch.no_grad():
    model.embedding.weight[PAD_ID].zero_()

# Clear previous gradients
model.zero_grad(set_to_none=True)

# Test again
model.train()

logits = model(source_ids, decoder_ids)

loss = F.cross_entropy(
    logits.reshape(-1, VOCAB_SIZE),
    target_ids.reshape(-1),
    ignore_index=-100
)

loss.backward()

print("Initial loss:", round(loss.item(), 4))

print(
    "Peak GPU memory:",
    round(
        torch.cuda.max_memory_allocated() / (1024 ** 3),
        3
    ),
    "GB"
)

print("Backward pass successful.")

Initial loss: 10.4619
Peak GPU memory: 0.174 GB
Backward pass successful.


In [19]:
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import json
import gc

# Training configuration
BATCH_SIZE = 8
EPOCHS = 1
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
SAVE_EVERY = 1000

# Load saved compact dataset
train_loader = DataLoader(
    compact_data["train"],
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_translation,
    num_workers=0,
    pin_memory=False
)

val_loader = DataLoader(
    compact_data["validation"],
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_translation,
    num_workers=0,
    pin_memory=False
)

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

best_val_loss = float("inf")
global_step = 0

# Training
for epoch in range(EPOCHS):

    model.train()
    total_train_loss = 0.0

    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for batch in progress:

        source_ids = batch["source_ids"].to(device)
        decoder_ids = batch["decoder_ids"].to(device)
        target_ids = batch["target_ids"].to(device)

        optimizer.zero_grad(set_to_none=True)

        logits = model(source_ids, decoder_ids)

        loss = F.cross_entropy(
            logits.reshape(-1, VOCAB_SIZE),
            target_ids.reshape(-1),
            ignore_index=-100
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        global_step += 1
        total_train_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

        # Save checkpoint periodically
        if global_step % SAVE_EVERY == 0:

            torch.save({
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "epoch": epoch + 1,
                "step": global_step,
                "vocab_size": VOCAB_SIZE,
                "d_model": D_MODEL
            }, MODEL_DIR / "last_checkpoint.pt")

            print(f"\nCheckpoint saved at step {global_step}")

    avg_train_loss = total_train_loss / len(train_loader)

    # Validation
    model.eval()

    validation_loss_sum = 0.0
    validation_token_count = 0

    with torch.no_grad():

        for batch in tqdm(val_loader, desc="Validation"):

            source_ids = batch["source_ids"].to(device)
            decoder_ids = batch["decoder_ids"].to(device)
            target_ids = batch["target_ids"].to(device)

            logits = model(source_ids, decoder_ids)

            loss_sum = F.cross_entropy(
                logits.reshape(-1, VOCAB_SIZE),
                target_ids.reshape(-1),
                ignore_index=-100,
                reduction="sum"
            )

            valid_tokens = (target_ids != -100).sum().item()

            validation_loss_sum += loss_sum.item()
            validation_token_count += valid_tokens

    avg_val_loss = validation_loss_sum / validation_token_count

    print(f"\nEpoch {epoch + 1} Results")
    print(f"Training loss:   {avg_train_loss:.4f}")
    print(f"Validation loss: {avg_val_loss:.4f}")

    # Save best model
    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": epoch + 1,
            "step": global_step,
            "validation_loss": best_val_loss,
            "vocab_size": VOCAB_SIZE,
            "d_model": D_MODEL
        }, MODEL_DIR / "best_model.pt")

        print("Best model checkpoint saved.")

    # Save final checkpoint
    torch.save({
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": epoch + 1,
        "step": global_step,
        "vocab_size": VOCAB_SIZE,
        "d_model": D_MODEL
    }, MODEL_DIR / "last_checkpoint.pt")

# Final GPU memory
print("\nTraining completed.")
print("Total training steps:", global_step)
print("Best validation loss:", round(best_val_loss, 4))

print(
    "Peak GPU memory:",
    round(
        torch.cuda.max_memory_allocated() / (1024 ** 3),
        3
    ),
    "GB"
)

print("\nSaved files:")
print("Best model:", MODEL_DIR / "best_model.pt")
print("Last checkpoint:", MODEL_DIR / "last_checkpoint.pt")

Epoch 1/1:   0%|          | 0/12500 [00:00<?, ?it/s]


Checkpoint saved at step 1000

Checkpoint saved at step 2000

Checkpoint saved at step 3000

Checkpoint saved at step 4000

Checkpoint saved at step 5000

Checkpoint saved at step 6000

Checkpoint saved at step 7000

Checkpoint saved at step 8000

Checkpoint saved at step 9000

Checkpoint saved at step 10000

Checkpoint saved at step 11000

Checkpoint saved at step 12000


Validation:   0%|          | 0/625 [00:00<?, ?it/s]

e:\punctuation_project\.venv\Lib\site-packages\torch\nn\modules\transformer.py:533: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:181.)
  output = torch._nested_tensor_from_mask(



Epoch 1 Results
Training loss:   5.1662
Validation loss: nan

Training completed.
Total training steps: 12500
Best validation loss: inf
Peak GPU memory: 0.603 GB

Saved files:
Best model: E:\punctuation_project\models\hindi_english_translation\best_model.pt
Last checkpoint: E:\punctuation_project\models\hindi_english_translation\last_checkpoint.pt


In [20]:
import torch
import torch.nn.functional as F

model.eval()

# 1. Check whether any trained model parameters are invalid
bad_parameters = []

for name, param in model.named_parameters():
    if not torch.isfinite(param).all():
        bad_parameters.append(name)

print("Non-finite model parameters:", bad_parameters)

# 2. Inspect validation batches
for batch_index, batch in enumerate(val_loader):

    source_ids = batch["source_ids"].to(device)
    decoder_ids = batch["decoder_ids"].to(device)
    target_ids = batch["target_ids"].to(device)

    with torch.no_grad():
        logits = model(source_ids, decoder_ids)

    valid_tokens = (target_ids != -100).sum().item()

    if (
        not torch.isfinite(logits).all()
        or valid_tokens == 0
    ):
        print("\nPROBLEM FOUND")
        print("Batch:", batch_index)
        print("Valid target tokens:", valid_tokens)
        print("Logits contain NaN:", torch.isnan(logits).any().item())
        print("Logits contain Inf:", torch.isinf(logits).any().item())
        print("Source shape:", tuple(source_ids.shape))
        print("Decoder shape:", tuple(decoder_ids.shape))
        print("Target shape:", tuple(target_ids.shape))
        break

    loss = F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),
        target_ids.reshape(-1),
        ignore_index=-100
    )

    if not torch.isfinite(loss):
        print("\nINVALID LOSS FOUND")
        print("Batch:", batch_index)
        print("Loss:", loss.item())
        break

else:
    print("\nAll validation batches produced finite logits and losses.")

Non-finite model parameters: []

PROBLEM FOUND
Batch: 0
Valid target tokens: 356
Logits contain NaN: True
Logits contain Inf: False
Source shape: (8, 128)
Decoder shape: (8, 114)
Target shape: (8, 114)


In [22]:
model.eval()

with torch.no_grad():
    test_logits = model(source_ids, decoder_ids)

print("Logits contain NaN:", torch.isnan(test_logits).any().item())
print("Logits contain Inf:", torch.isinf(test_logits).any().item())

test_loss = F.cross_entropy(
    test_logits.reshape(-1, VOCAB_SIZE),
    target_ids.reshape(-1),
    ignore_index=-100
)

print("Validation batch loss:", test_loss.item())

Logits contain NaN: False
Logits contain Inf: False
Validation batch loss: 77.17916107177734


In [23]:
# Check the numerical range of model predictions
print("Logits:")
print("  Minimum:", test_logits.min().item())
print("  Maximum:", test_logits.max().item())
print("  Mean:", test_logits.mean().item())

# Calculate loss for each target token
per_token_loss = F.cross_entropy(
    test_logits.reshape(-1, VOCAB_SIZE),
    target_ids.reshape(-1),
    ignore_index=-100,
    reduction="none"
)

valid_mask = target_ids.reshape(-1) != -100
valid_losses = per_token_loss[valid_mask]

print("\nToken-level loss:")
print("  Mean:", valid_losses.mean().item())
print("  Minimum:", valid_losses.min().item())
print("  Maximum:", valid_losses.max().item())

# Check prediction accuracy
predictions = test_logits.argmax(dim=-1)
correct = (predictions == target_ids) & (target_ids != -100)

print("\nToken accuracy:", 
      (correct.sum().item() / valid_mask.sum().item()) * 100, "%")

print("Valid target tokens:", valid_mask.sum().item())
print("Target ID range:",
      target_ids[valid_mask.reshape(target_ids.shape)].min().item(),
      "to",
      target_ids[valid_mask.reshape(target_ids.shape)].max().item())

Logits:
  Minimum: -59.92213439941406
  Maximum: 108.1656723022461
  Mean: 0.007452731486409903

Token-level loss:
  Mean: 77.17916107177734
  Minimum: 30.356809616088867
  Maximum: 121.42308044433594

Token accuracy: 0.0 %
Valid target tokens: 356
Target ID range: 1 to 25783


In [25]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google/mt5-small")

print("Tokenizer loaded.")
print("Vocabulary size:", tokenizer.vocab_size)

Tokenizer loaded.
Vocabulary size: 250100


In [26]:
# Select the first validation example
i = 0

# Convert compact token IDs back to the original tokenizer IDs
source_original = [
    compact_to_original[int(x)]
    for x in source_ids[i].tolist()
    if int(x) != PAD_ID
]

target_original = [
    compact_to_original[int(x)]
    for x in target_ids[i].tolist()
    if int(x) not in [-100, PAD_ID]
]

predicted_ids = test_logits[i].argmax(dim=-1).tolist()

predicted_original = [
    compact_to_original[int(x)]
    for x in predicted_ids
    if int(x) not in [PAD_ID]
]

print("SOURCE (Hindi):")
print(tokenizer.decode(source_original, skip_special_tokens=True))

print("\nEXPECTED TARGET (English):")
print(tokenizer.decode(target_original, skip_special_tokens=True))

print("\nMODEL PREDICTION:")
print(tokenizer.decode(predicted_original, skip_special_tokens=True))

SOURCE (Hindi):
translate Hindi to English: कांग्रेस के सभाZपति मौलाना अबुल कलाम आज़ाद और गांधी जी ने आखिर में अपने भाषणों में यह खुलासा कर दिया कि वह पहले वायसराय से मिलेंगे, जो ब्रिटिश सरकार के नुमांइदा हैं. इसके अलावा वह राष्ट्र संघ के सदस्य खास खास मुल्कों के अध्यक्षों से अपील करेंगे, जिससे कोई ऐसा समझौता हो सके, जो दोनों तरफ के लोगों को मान्य

EXPECTED TARGET (English):
In their concluding speeches Maulana 'Abul Kalam Azad, the Congress president, and Gandhiji made it clear that their next step would be to approach the Viceroy, as representing the British Government, and to appeal to the heads of the principal United Nations for an honourable settlement, which while recognizing the freedom of India, would also advance the cause of the United Nations in the struggle against the aggressor Axis powers.

MODEL PREDICTION:
conference In their concluding speeches Maulana 'Abul Kalam Azad, the Congress president, and Gandhiji made it clear that their next step would be to approach the Vi

In [27]:
print("DECODER INPUT:")
print(tokenizer.decode(
    [compact_to_original[int(x)] for x in decoder_ids[0].tolist()
     if int(x) != PAD_ID],
    skip_special_tokens=True
))

print("\nEXPECTED TARGET:")
print(tokenizer.decode(
    [compact_to_original[int(x)] for x in target_ids[0].tolist()
     if int(x) not in [-100, PAD_ID]],
    skip_special_tokens=True
))

print("\nFIRST 15 DECODER IDs:")
print(decoder_ids[0][:15].tolist())

print("\nFIRST 15 TARGET IDs:")
print(target_ids[0][:15].tolist())

print("\nFIRST 15 PREDICTED IDs:")
print(test_logits[0].argmax(dim=-1)[:15].tolist())

DECODER INPUT:
In their concluding speeches Maulana 'Abul Kalam Azad, the Congress president, and Gandhiji made it clear that their next step would be to approach the Viceroy, as representing the British Government, and to appeal to the heads of the principal United Nations for an honourable settlement, which while recognizing the freedom of India, would also advance the cause of the United Nations in the struggle against the aggressor Axis powers.

EXPECTED TARGET:
In their concluding speeches Maulana 'Abul Kalam Azad, the Congress president, and Gandhiji made it clear that their next step would be to approach the Viceroy, as representing the British Government, and to appeal to the heads of the principal United Nations for an honourable settlement, which while recognizing the freedom of India, would also advance the cause of the United Nations in the struggle against the aggressor Axis powers.

FIRST 15 DECODER IDs:
[0, 308, 3, 158, 8347, 5199, 3, 3135, 182, 11879, 4961, 3, 56, 1292,

In [28]:
print("Decoder first 10:", decoder_ids[0, :10].tolist())
print("Target first 10: ", target_ids[0, :10].tolist())

print("\nAre decoder IDs shifted right?")
print(
    "Decoder[1:] == Target[:-1]:",
    torch.equal(
        decoder_ids[0, 1:],
        target_ids[0, :-1]
    )
)

Decoder first 10: [0, 308, 3, 158, 8347, 5199, 3, 3135, 182, 11879]
Target first 10:  [308, 3, 158, 8347, 5199, 3, 3135, 182, 11879, 4961]

Are decoder IDs shifted right?
Decoder[1:] == Target[:-1]: True


In [29]:
predicted_ids = test_logits.argmax(dim=-1)

target = target_ids[0]
predicted = predicted_ids[0]

print("Correct at same position:",
      (predicted == target).sum().item())

print("Correct at next position:",
      (predicted[:-1] == target[1:]).sum().item())

print("Correct at previous position:",
      (predicted[1:] == target[:-1]).sum().item())

print("\nFirst 20 predictions:")
print(predicted[:20].tolist())

print("\nFirst 20 targets:")
print(target[:20].tolist())

Correct at same position: 0
Correct at next position: 3
Correct at previous position: 113

First 20 predictions:
[4794, 308, 3, 158, 8347, 5199, 3, 3135, 182, 11879, 4961, 3, 56, 1292, 13153, 12593, 7549, 3871, 9, 10]

First 20 targets:
[308, 3, 158, 8347, 5199, 3, 3135, 182, 11879, 4961, 3, 56, 1292, 13153, 12593, 7549, 3871, 9, 10, 2128]


In [30]:
# Compare the score assigned to the previous input token
# against the score assigned to the correct next token.

for pos in range(1, 11):

    logits_at_pos = test_logits[0, pos]

    previous_token = decoder_ids[0, pos].item()
    expected_token = target_ids[0, pos].item()

    previous_score = logits_at_pos[previous_token].item()
    expected_score = logits_at_pos[expected_token].item()

    print(f"\nPosition {pos}")
    print("Previous input token:", previous_token)
    print("Expected next token:", expected_token)
    print("Previous token score:", round(previous_score, 3))
    print("Expected token score:", round(expected_score, 3))
    print("Top prediction:", logits_at_pos.argmax().item())


Position 1
Previous input token: 308
Expected next token: 3
Previous token score: 79.215
Expected token score: 17.968
Top prediction: 308

Position 2
Previous input token: 3
Expected next token: 158
Previous token score: 74.041
Expected token score: 3.706
Top prediction: 3

Position 3
Previous input token: 158
Expected next token: 8347
Previous token score: 93.75
Expected token score: 8.057
Top prediction: 158

Position 4
Previous input token: 8347
Expected next token: 5199
Previous token score: 76.261
Expected token score: 6.711
Top prediction: 8347

Position 5
Previous input token: 5199
Expected next token: 3
Previous token score: 72.259
Expected token score: -1.681
Top prediction: 5199

Position 6
Previous input token: 3
Expected next token: 3135
Previous token score: 60.492
Expected token score: 5.621
Top prediction: 3

Position 7
Previous input token: 3135
Expected next token: 182
Previous token score: 87.969
Expected token score: -1.932
Top prediction: 3135

Position 8
Previous 

In [31]:
checkpoint_path = MODEL_DIR / "last_checkpoint.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=True
)

print("CHECKPOINT DETAILS")
print("Saved epoch:", checkpoint.get("epoch"))
print("Saved training step:", checkpoint.get("step"))
print("Saved vocabulary size:", checkpoint.get("vocab_size"))
print("Saved model dimension:", checkpoint.get("d_model"))

print("\nCHECKPOINT EMBEDDING SHAPE:")
print(
    checkpoint["model_state_dict"]["embedding.weight"].shape
)

print("\nCURRENT MODEL EMBEDDING SHAPE:")
print(model.embedding.weight.shape)

CHECKPOINT DETAILS
Saved epoch: 1
Saved training step: 12500
Saved vocabulary size: 32000
Saved model dimension: 128

CHECKPOINT EMBEDDING SHAPE:
torch.Size([32000, 128])

CURRENT MODEL EMBEDDING SHAPE:
torch.Size([32000, 128])


In [32]:
# Load the saved training checkpoint
checkpoint = torch.load(
    MODEL_DIR / "last_checkpoint.pt",
    map_location=device,
    weights_only=True
)

# Restore trained weights into the existing model
model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("Trained model restored successfully.")
print("Epoch:", checkpoint["epoch"])
print("Training steps:", checkpoint["step"])
print("Vocabulary size:", checkpoint["vocab_size"])

# Verify that model weights match the checkpoint
print(
    "Weights match checkpoint:",
    torch.equal(
        model.embedding.weight,
        checkpoint["model_state_dict"]["embedding.weight"]
    )
)

Trained model restored successfully.
Epoch: 1
Training steps: 12500
Vocabulary size: 32000
Weights match checkpoint: True


In [33]:
model.eval()

with torch.no_grad():
    test_logits = model(source_ids, decoder_ids)

print("NaN:", torch.isnan(test_logits).any().item())
print("Inf:", torch.isinf(test_logits).any().item())

test_loss = F.cross_entropy(
    test_logits.reshape(-1, VOCAB_SIZE),
    target_ids.reshape(-1),
    ignore_index=-100
)

print("Validation batch loss:", test_loss.item())

NaN: False
Inf: False
Validation batch loss: 4.559892177581787


In [34]:
model.eval()

# Take the first validation example
source_sample = source_ids[0:1]

generated_ids = [START_ID]

with torch.no_grad():
    for _ in range(MAX_TARGET_LENGTH):
        decoder_input = torch.tensor(
            [generated_ids],
            dtype=torch.long,
            device=device
        )

        logits = model(source_sample, decoder_input)

        next_token = logits[0, -1, :].argmax().item()

        if next_token == EOS_ID:
            break

        generated_ids.append(next_token)


# Convert compact vocabulary IDs back to original tokenizer IDs
def convert_to_original(compact_ids):
    if isinstance(compact_to_original, dict):
        return [
            compact_to_original.get(str(i),
                compact_to_original.get(i, UNK_ID))
            for i in compact_ids
        ]

    return [compact_to_original[i] for i in compact_ids]


# Decode source, expected translation, and generated translation
source_original = convert_to_original(
    source_sample[0].tolist()
)

target_original = convert_to_original([
    int(i) for i in target_ids[0].tolist()
    if i != -100
])

prediction_original = convert_to_original(generated_ids[1:])

print("HINDI INPUT:")
print(tokenizer.decode(source_original, skip_special_tokens=True))

print("\nEXPECTED ENGLISH:")
print(tokenizer.decode(target_original, skip_special_tokens=True))

print("\nMODEL PREDICTION:")
print(tokenizer.decode(prediction_original, skip_special_tokens=True))

HINDI INPUT:
translate Hindi to English: कांग्रेस के सभाZपति मौलाना अबुल कलाम आज़ाद और गांधी जी ने आखिर में अपने भाषणों में यह खुलासा कर दिया कि वह पहले वायसराय से मिलेंगे, जो ब्रिटिश सरकार के नुमांइदा हैं. इसके अलावा वह राष्ट्र संघ के सदस्य खास खास मुल्कों के अध्यक्षों से अपील करेंगे, जिससे कोई ऐसा समझौता हो सके, जो दोनों तरफ के लोगों को मान्य

EXPECTED ENGLISH:
In their concluding speeches Maulana 'Abul Kalam Azad, the Congress president, and Gandhiji made it clear that their next step would be to approach the Viceroy, as representing the British Government, and to appeal to the heads of the principal United Nations for an honourable settlement, which while recognizing the freedom of India, would also advance the cause of the United Nations in the struggle against the aggressor Axis powers.

MODEL PREDICTION:
The President, the President will be a a British Government would be a a a British Government of the which the would be a a British Government of the which the would be would be

In [35]:
# Compare model predictions with expected target tokens
predicted_ids = test_logits.argmax(dim=-1)

valid_positions = target_ids != -100

correct_tokens = (
    (predicted_ids == target_ids) & valid_positions
).sum().item()

total_tokens = valid_positions.sum().item()

accuracy = correct_tokens / total_tokens * 100

print("Correct tokens:", correct_tokens)
print("Total valid tokens:", total_tokens)
print(f"Teacher-forced token accuracy: {accuracy:.2f}%")

Correct tokens: 90
Total valid tokens: 356
Teacher-forced token accuracy: 25.28%


In [36]:

from datasets import load_from_disk
from torch.utils.data import DataLoader
from pathlib import Path
import torch
import torch.nn.functional as F

# Load the existing compact validation dataset
DATA_DIR = Path(r"E:\punctuation_project\data\processed\translation_compact_32k_len128")

translation_data = load_from_disk(str(DATA_DIR))
validation_dataset = translation_data["validation"]

print("Validation examples:", len(validation_dataset))


# Prepare batches with padding
def validation_collate_fn(batch):
    batch_size = len(batch)

    max_source_len = max(len(item["source_ids"]) for item in batch)
    max_target_len = max(len(item["target_ids"]) for item in batch)

    source_batch = torch.full(
        (batch_size, max_source_len), PAD_ID, dtype=torch.long
    )

    target_batch = torch.full(
        (batch_size, max_target_len), -100, dtype=torch.long
    )

    decoder_batch = torch.full(
        (batch_size, max_target_len), PAD_ID, dtype=torch.long
    )

    for i, item in enumerate(batch):
        source = item["source_ids"]
        target = item["target_ids"]

        source_batch[i, :len(source)] = torch.tensor(source)
        target_batch[i, :len(target)] = torch.tensor(target)

        decoder_sequence = [START_ID] + target[:-1]
        decoder_batch[i, :len(decoder_sequence)] = torch.tensor(
            decoder_sequence
        )

    return source_batch, decoder_batch, target_batch


validation_loader = DataLoader(
    validation_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=validation_collate_fn,
    num_workers=0
)


# Evaluate the restored model
model.eval()
model.to(device)

total_loss = 0.0
total_correct = 0
total_tokens = 0

with torch.no_grad():
    for source_batch, decoder_batch, target_batch in validation_loader:

        source_batch = source_batch.to(device)
        decoder_batch = decoder_batch.to(device)
        target_batch = target_batch.to(device)

        logits = model(source_batch, decoder_batch)

        loss_sum = F.cross_entropy(
            logits.reshape(-1, VOCAB_SIZE),
            target_batch.reshape(-1),
            ignore_index=-100,
            reduction="sum"
        )

        predictions = logits.argmax(dim=-1)
        valid_positions = target_batch != -100

        total_loss += loss_sum.item()

        total_correct += (
            (predictions == target_batch) & valid_positions
        ).sum().item()

        total_tokens += valid_positions.sum().item()


# Final metrics
average_loss = total_loss / total_tokens
token_accuracy = (total_correct / total_tokens) * 100

print("\n========== FULL VALIDATION RESULTS ==========")
print("Validation examples:", len(validation_dataset))
print("Total valid tokens:", total_tokens)
print(f"Average validation loss: {average_loss:.4f}")
print(f"Teacher-forced token accuracy: {token_accuracy:.2f}%")
print("=============================================")


Validation examples: 5000

========== FULL VALIDATION RESULTS ==========
Validation examples: 5000
Total valid tokens: 103081
Average validation loss: 4.5815
Teacher-forced token accuracy: 26.98%


In [37]:
checkpoint = torch.load(
    MODEL_DIR / "last_checkpoint.pt",
    map_location=device,
    weights_only=True
)

# Create optimizer for the restored model
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.01
)

# Restore optimizer's previous training state
optimizer.load_state_dict(
    checkpoint["optimizer_state_dict"]
)

print("Optimizer restored successfully.")
print("Previous epoch:", checkpoint["epoch"])
print("Previous training steps:", checkpoint["step"])
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Optimizer state entries:", len(optimizer.state))

Optimizer restored successfully.
Previous epoch: 1
Previous training steps: 12500
Learning rate: 0.0003
Optimizer state entries: 63


In [39]:
print("Train loader type:", type(train_loader))

first_batch = next(iter(train_loader))

print("Batch type:", type(first_batch))

if isinstance(first_batch, dict):
    print("Batch keys:", first_batch.keys())

    for key, value in first_batch.items():
        print(
            key,
            "| Type:", type(value),
            "| Shape:", getattr(value, "shape", None)
        )

elif isinstance(first_batch, (tuple, list)):
    for i, value in enumerate(first_batch):
        print(
            i,
            "| Type:", type(value),
            "| Shape:", getattr(value, "shape", None)
        )

Train loader type: <class 'torch.utils.data.dataloader.DataLoader'>
Batch type: <class 'dict'>
Batch keys: dict_keys(['source_ids', 'decoder_ids', 'target_ids'])
source_ids | Type: <class 'torch.Tensor'> | Shape: torch.Size([8, 112])
decoder_ids | Type: <class 'torch.Tensor'> | Shape: torch.Size([8, 74])
target_ids | Type: <class 'torch.Tensor'> | Shape: torch.Size([8, 74])


In [40]:

import time
import torch
import torch.nn.functional as F

model.train()

epoch_loss = 0.0
start_time = time.time()

for step, batch in enumerate(train_loader, start=1):

    source_ids = batch["source_ids"].to(device)
    decoder_ids = batch["decoder_ids"].to(device)
    target_ids = batch["target_ids"].to(device)

    optimizer.zero_grad(set_to_none=True)

    logits = model(source_ids, decoder_ids)

    loss = F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),
        target_ids.reshape(-1),
        ignore_index=-100
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        model.parameters(), max_norm=1.0
    )

    optimizer.step()

    epoch_loss += loss.item()

    if step % 1000 == 0:
        elapsed = time.time() - start_time

        print(
            f"Epoch 2 | Step {step}/{len(train_loader)} "
            f"| Loss: {loss.item():.4f} "
            f"| Time: {elapsed/60:.1f} min"
        )

average_train_loss = epoch_loss / len(train_loader)

print("\n========== EPOCH 2 COMPLETE ==========")
print(f"Average training loss: {average_train_loss:.4f}")
print(f"Total steps completed: {len(train_loader)}")


Epoch 2 | Step 1000/12500 | Loss: 4.5998 | Time: 0.8 min
Epoch 2 | Step 2000/12500 | Loss: 3.9646 | Time: 1.6 min
Epoch 2 | Step 3000/12500 | Loss: 4.7069 | Time: 2.3 min
Epoch 2 | Step 4000/12500 | Loss: 4.2252 | Time: 3.1 min
Epoch 2 | Step 5000/12500 | Loss: 4.7228 | Time: 3.8 min
Epoch 2 | Step 6000/12500 | Loss: 4.0481 | Time: 4.6 min
Epoch 2 | Step 7000/12500 | Loss: 4.6263 | Time: 5.3 min
Epoch 2 | Step 8000/12500 | Loss: 5.0558 | Time: 6.1 min
Epoch 2 | Step 9000/12500 | Loss: 4.0860 | Time: 6.8 min
Epoch 2 | Step 10000/12500 | Loss: 4.5082 | Time: 7.6 min
Epoch 2 | Step 11000/12500 | Loss: 3.8689 | Time: 8.4 min
Epoch 2 | Step 12000/12500 | Loss: 4.6219 | Time: 9.1 min

========== EPOCH 2 COMPLETE ==========
Average training loss: 4.3503
Total steps completed: 12500


In [41]:
checkpoint_path = MODEL_DIR / "epoch2_checkpoint.pt"

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "epoch": 2,
        "step": 25000,
        "vocab_size": VOCAB_SIZE,
        "d_model": D_MODEL,
    },
    checkpoint_path
)

print("Epoch 2 checkpoint saved:", checkpoint_path)
print("File exists:", checkpoint_path.exists())

Epoch 2 checkpoint saved: E:\punctuation_project\models\hindi_english_translation\epoch2_checkpoint.pt
File exists: True


In [43]:
model.eval()

total_loss = 0.0
total_correct = 0
total_tokens = 0

with torch.no_grad():
    for batch in validation_loader:

        source_batch, decoder_batch, target_batch = batch

        source_batch = source_batch.to(device)
        decoder_batch = decoder_batch.to(device)
        target_batch = target_batch.to(device)

        logits = model(source_batch, decoder_batch)

        loss_sum = F.cross_entropy(
            logits.reshape(-1, VOCAB_SIZE),
            target_batch.reshape(-1),
            ignore_index=-100,
            reduction="sum"
        )

        predictions = logits.argmax(dim=-1)
        valid_positions = target_batch != -100

        total_loss += loss_sum.item()

        total_correct += (
            (predictions == target_batch) & valid_positions
        ).sum().item()

        total_tokens += valid_positions.sum().item()


epoch2_val_loss = total_loss / total_tokens
epoch2_accuracy = total_correct / total_tokens * 100

print("\n========== EPOCH 2 VALIDATION ==========")
print("Validation examples:", len(validation_dataset))
print("Total valid tokens:", total_tokens)
print(f"Average validation loss: {epoch2_val_loss:.4f}")
print(f"Teacher-forced token accuracy: {epoch2_accuracy:.2f}%")
print("=======================================")


========== EPOCH 2 VALIDATION ==========
Validation examples: 5000
Total valid tokens: 103081
Average validation loss: 4.0816
Teacher-forced token accuracy: 32.11%


In [44]:
model.eval()

# Get the first validation example
sample = validation_dataset[0]

source_sample = torch.tensor(
    [sample["source_ids"]],
    dtype=torch.long,
    device=device
)

generated_ids = [START_ID]

with torch.no_grad():
    for _ in range(MAX_TARGET_LENGTH):
        decoder_input = torch.tensor(
            [generated_ids],
            dtype=torch.long,
            device=device
        )

        logits = model(source_sample, decoder_input)

        next_token = logits[0, -1, :].argmax().item()

        if next_token == EOS_ID:
            break

        generated_ids.append(next_token)


# Convert compact IDs to original tokenizer IDs
prediction_original = convert_to_original(generated_ids[1:])

source_original = convert_to_original(sample["source_ids"])
target_original = convert_to_original(sample["target_ids"])

print("HINDI INPUT:")
print(tokenizer.decode(source_original, skip_special_tokens=True))

print("\nEXPECTED ENGLISH:")
print(tokenizer.decode(target_original, skip_special_tokens=True))

print("\nEPOCH 2 PREDICTION:")
print(tokenizer.decode(prediction_original, skip_special_tokens=True))

HINDI INPUT:
translate Hindi to English: कांग्रेस के सभाZपति मौलाना अबुल कलाम आज़ाद और गांधी जी ने आखिर में अपने भाषणों में यह खुलासा कर दिया कि वह पहले वायसराय से मिलेंगे, जो ब्रिटिश सरकार के नुमांइदा हैं. इसके अलावा वह राष्ट्र संघ के सदस्य खास खास मुल्कों के अध्यक्षों से अपील करेंगे, जिससे कोई ऐसा समझौता हो सके, जो दोनों तरफ के लोगों को मान्य

EXPECTED ENGLISH:
In their concluding speeches Maulana 'Abul Kalam Azad, the Congress president, and Gandhiji made it clear that their next step would be to approach the Viceroy, as representing the British Government, and to appeal to the heads of the principal United Nations for an honourable settlement, which while recognizing the freedom of India, would also advance the cause of the United Nations in the struggle against the aggressor Axis powers.

EPOCH 2 PREDICTION:
The Congress would be taken to the Congress and the Congress Gandhiji would be taken to the British Government and the British Governments of the Congress British Governments,

In [45]:
# Inspect EOS predictions during teacher-forced validation

model.eval()

eos_id = EOS_ID
eos_correct = 0
eos_total = 0
eos_predicted = 0

with torch.no_grad():
    for batch in validation_loader:

        source_batch, decoder_batch, target_batch = batch

        source_batch = source_batch.to(device)
        decoder_batch = decoder_batch.to(device)
        target_batch = target_batch.to(device)

        logits = model(source_batch, decoder_batch)
        predictions = logits.argmax(dim=-1)

        valid_positions = target_batch != -100
        eos_positions = (target_batch == eos_id) & valid_positions

        eos_correct += (
            (predictions == eos_id) & eos_positions
        ).sum().item()

        eos_total += eos_positions.sum().item()

        eos_predicted += (
            (predictions == eos_id) & valid_positions
        ).sum().item()


print("EOS tokens in validation:", eos_total)
print("Correct EOS predictions:", eos_correct)
print("EOS recall:", round(eos_correct / max(eos_total, 1) * 100, 2), "%")
print("Total EOS predictions:", eos_predicted)

EOS tokens in validation: 5000
Correct EOS predictions: 4592
EOS recall: 91.84 %
Total EOS predictions: 6572


In [46]:

model.eval()

position_correct = {}
position_total = {}

with torch.no_grad():
    for batch in validation_loader:
        source_batch, decoder_batch, target_batch = batch

        source_batch = source_batch.to(device)
        decoder_batch = decoder_batch.to(device)
        target_batch = target_batch.to(device)

        logits = model(source_batch, decoder_batch)
        predictions = logits.argmax(dim=-1)

        valid = target_batch != -100
        correct = (predictions == target_batch) & valid

        for pos in range(target_batch.size(1)):
            position_correct[pos] = (
                position_correct.get(pos, 0)
                + correct[:, pos].sum().item()
            )
            position_total[pos] = (
                position_total.get(pos, 0)
                + valid[:, pos].sum().item()
            )

print("POSITION-WISE TOKEN ACCURACY")

for start in range(0, 128, 16):
    end = min(start + 16, 128)

    total_correct = sum(position_correct.get(p, 0) for p in range(start, end))
    total_tokens = sum(position_total.get(p, 0) for p in range(start, end))

    accuracy = 100 * total_correct / max(total_tokens, 1)

    print(f"Positions {start:3d}-{end-1:3d}: {accuracy:.2f}%")


POSITION-WISE TOKEN ACCURACY
Positions   0- 15: 33.37%
Positions  16- 31: 31.17%
Positions  32- 47: 31.32%
Positions  48- 63: 30.33%
Positions  64- 79: 27.28%
Positions  80- 95: 25.71%
Positions  96-111: 24.09%
Positions 112-127: 29.36%


In [50]:

model.eval()

def generate_translation(source_ids, max_length=128):
    source_ids = source_ids.unsqueeze(0).to(device)

    generated = torch.tensor(
        [[START_ID]], dtype=torch.long, device=device
    )

    with torch.no_grad():
        source_length = source_ids.size(1)
        source_positions = torch.arange(
            source_length, device=device
        ).unsqueeze(0)

        source = model.dropout(
            model.embedding(source_ids)
            + model.source_position(source_positions)
        )

        source_padding_mask = source_ids.eq(PAD_ID)

        memory = model.encoder(
            source,
            src_key_padding_mask=source_padding_mask
        )

        for _ in range(max_length):
            target_length = generated.size(1)

            target_positions = torch.arange(
                target_length, device=device
            ).unsqueeze(0)

            target = model.dropout(
                model.embedding(generated)
                + model.target_position(target_positions)
            )

            causal_mask = torch.triu(
                torch.ones(
                    target_length, target_length,
                    device=device, dtype=torch.bool
                ),
                diagonal=1
            )

            decoded = model.decoder(
                tgt=target,
                memory=memory,
                tgt_mask=causal_mask,
                memory_key_padding_mask=source_padding_mask
            )

            logits = F.linear(decoded, model.embedding.weight)

            next_token = logits[:, -1, :].argmax(dim=-1).item()

            if next_token == EOS_ID:
                break

            generated = torch.cat(
                [generated, torch.tensor(
                    [[next_token]], device=device
                )],
                dim=1
            )

    compact_ids = generated[0].tolist()

    original_ids = [
        compact_to_original[token_id]
        for token_id in compact_ids
    ]

    return tokenizer.decode(
        original_ids,
        skip_special_tokens=True
    )


# Inspect five validation examples


# Inspect five validation examples
for i in range(5):
    example = validation_dataset[i]

    source_ids = example["source_ids"]
    target_ids = example["target_ids"]

    source_text = tokenizer.decode(
        [compact_to_original[x] for x in source_ids
         if x != PAD_ID],
        skip_special_tokens=True
    )

    expected_ids = [
        compact_to_original[x]
        for x in target_ids
        if x not in [-100, PAD_ID, EOS_ID]
    ]

    expected_text = tokenizer.decode(
        expected_ids,
        skip_special_tokens=True
    )

    predicted_text = generate_translation(
        torch.tensor(source_ids, dtype=torch.long)
    )

    print(f"\n{'='*70}")
    print(f"EXAMPLE {i+1}")
    print("HINDI:     ", source_text)
    print("EXPECTED:  ", expected_text)
    print("PREDICTED: ", predicted_text)



EXAMPLE 1
HINDI:      translate Hindi to English: कांग्रेस के सभाZपति मौलाना अबुल कलाम आज़ाद और गांधी जी ने आखिर में अपने भाषणों में यह खुलासा कर दिया कि वह पहले वायसराय से मिलेंगे, जो ब्रिटिश सरकार के नुमांइदा हैं. इसके अलावा वह राष्ट्र संघ के सदस्य खास खास मुल्कों के अध्यक्षों से अपील करेंगे, जिससे कोई ऐसा समझौता हो सके, जो दोनों तरफ के लोगों को मान्य
EXPECTED:   In their concluding speeches Maulana 'Abul Kalam Azad, the Congress president, and Gandhiji made it clear that their next step would be to approach the Viceroy, as representing the British Government, and to appeal to the heads of the principal United Nations for an honourable settlement, which while recognizing the freedom of India, would also advance the cause of the United Nations in the struggle against the aggressor Axis powers.
PREDICTED:  The Congress would be taken to the Congress and the Congress Gandhiji would be taken to the British Government and the British Governments of the Congress British Governments, which

In [51]:

model.eval()

for i in range(5):
    example = validation_dataset[i]

    source_ids = example["source_ids"]
    target_ids = example["target_ids"]

    # Give the model the correct previous target tokens.
    decoder_ids = [START_ID] + target_ids[:-1]

    source_tensor = torch.tensor(
        [source_ids], dtype=torch.long, device=device
    )

    decoder_tensor = torch.tensor(
        [decoder_ids], dtype=torch.long, device=device
    )

    with torch.no_grad():
        logits = model(source_tensor, decoder_tensor)
        predicted_ids = logits.argmax(dim=-1)[0].tolist()

    # Convert compact vocabulary IDs back to tokenizer IDs.
    predicted_original_ids = []

    for token_id in predicted_ids:
        if token_id == EOS_ID:
            break
        if token_id != PAD_ID:
            predicted_original_ids.append(
                compact_to_original[token_id]
            )

    expected_original_ids = [
        compact_to_original[token_id]
        for token_id in target_ids
        if token_id not in [PAD_ID, EOS_ID]
    ]

    teacher_forced_text = tokenizer.decode(
        predicted_original_ids,
        skip_special_tokens=True
    )

    expected_text = tokenizer.decode(
        expected_original_ids,
        skip_special_tokens=True
    )

    print(f"\n{'='*70}")
    print(f"EXAMPLE {i+1}")
    print("EXPECTED:      ", expected_text)
    print("TEACHER-FORCED:", teacher_forced_text)



EXAMPLE 1
EXPECTED:       In their concluding speeches Maulana 'Abul Kalam Azad, the Congress president, and Gandhiji made it clear that their next step would be to approach the Viceroy, as representing the British Government, and to appeal to the heads of the principal United Nations for an honourable settlement, which while recognizing the freedom of India, would also advance the cause of the United Nations in the struggle against the aggressor Axis powers.
TEACHER-FORCED: The thea de Gandhi,,ana,'s,t,ad, the President and, the thewouldji  the ly thea   inrights be  be to  President of  theing the British Government of  the thel  the British of the   StatesBritish. the ur of British of of British theBritish theBritishe the British of the. British be bes British of the  StatesBritish. the uggle.British the ressorss ofs of

EXAMPLE 2
EXPECTED:       The revolutionary prisoners in Bengal and 'where who were not included in the general amnesty granted to satyagrahi prisoners as a result

In [52]:

# Check whether compact vocabulary mapping is consistent

example = validation_dataset[0]

source_ids = example["source_ids"]
target_ids = example["target_ids"]

print("COMPACT VOCABULARY CHECK")
print("Compact vocabulary size:", len(compact_to_original))
print("Model vocabulary size:", model.embedding.num_embeddings)
print("Mapping size:", len(original_to_compact))

# Check that the mapping works in both directions.
mapping_errors = 0

for compact_id, original_id in enumerate(compact_to_original):
    if original_to_compact.get(original_id) != compact_id:
        mapping_errors += 1

print("Mapping inconsistencies:", mapping_errors)

# Check that all dataset IDs are valid.
all_ids = source_ids + target_ids

invalid_ids = [
    token_id for token_id in all_ids
    if token_id < 0 or token_id >= len(compact_to_original)
]

print("Invalid dataset token IDs:", len(invalid_ids))

# Decode the actual stored target sequence.
original_target_ids = [
    compact_to_original[token_id]
    for token_id in target_ids
]

print("\nTARGET ROUND-TRIP CHECK")
print("Decoded target:")
print(tokenizer.decode(
    original_target_ids,
    skip_special_tokens=True
))

print("\nFIRST 25 TARGET TOKENS")
for compact_id in target_ids[:25]:
    original_id = compact_to_original[compact_id]
    token = tokenizer.convert_ids_to_tokens(original_id)

    print(
        f"Compact ID: {compact_id:5d} | "
        f"Original ID: {original_id:6d} | "
        f"Token: {token}"
    )


COMPACT VOCABULARY CHECK
Compact vocabulary size: 32000
Model vocabulary size: 32000
Mapping size: 32000
Mapping inconsistencies: 0
Invalid dataset token IDs: 0

TARGET ROUND-TRIP CHECK
Decoded target:
In their concluding speeches Maulana 'Abul Kalam Azad, the Congress president, and Gandhiji made it clear that their next step would be to approach the Viceroy, as representing the British Government, and to appeal to the heads of the principal United Nations for an honourable settlement, which while recognizing the freedom of India, would also advance the cause of the United Nations in the struggle against the aggressor Axis powers.

FIRST 25 TARGET TOKENS
Compact ID:   308 | Original ID:    563 | Token: ▁In
Compact ID:     3 | Original ID:    259 | Token: ▁
Compact ID:   158 | Original ID:   1616 | Token: their
Compact ID:  8347 | Original ID:  21377 | Token: ▁conclu
Compact ID:  5199 | Original ID:  10646 | Token: ding
Compact ID:     3 | Original ID:    259 | Token: ▁
Compact ID:  31

In [54]:

from datasets import load_from_disk

translation_data = load_from_disk(
    r"E:\punctuation_project\data\processed\translation_compact_32k_len128"
)

print("Available splits:", translation_data.keys())
print("Training examples:", len(translation_data["train"]))

# Use the existing training split
train_dataset = translation_data["train"]

print("\nFirst training example:")
print("Source IDs:", train_dataset[0]["source_ids"][:10])
print("Target IDs:", train_dataset[0]["target_ids"][:10])


Available splits: dict_keys(['train', 'validation'])
Training examples: 100000

First training example:
Source IDs: [8, 6, 4, 7, 5, 1408, 4018, 31, 3, 1902]
Target IDs: [24361, 61, 10, 4793, 13, 2343, 368, 11559, 3, 56]


In [55]:

model.eval()

for i in range(5):
    example = train_dataset[i]

    source_ids = example["source_ids"]
    target_ids = example["target_ids"]

    decoder_ids = [START_ID] + target_ids[:-1]

    source_tensor = torch.tensor(
        [source_ids], dtype=torch.long, device=device
    )

    decoder_tensor = torch.tensor(
        [decoder_ids], dtype=torch.long, device=device
    )

    with torch.no_grad():
        logits = model(source_tensor, decoder_tensor)
        predicted_ids = logits.argmax(dim=-1)[0].tolist()

    predicted_original_ids = []

    for token_id in predicted_ids:
        if token_id == EOS_ID:
            break
        if token_id != PAD_ID:
            predicted_original_ids.append(
                compact_to_original[token_id]
            )

    expected_original_ids = [
        compact_to_original[token_id]
        for token_id in target_ids
        if token_id not in [PAD_ID, EOS_ID]
    ]

    print(f"\n{'='*70}")
    print(f"TRAINING EXAMPLE {i+1}")

    print("EXPECTED:")
    print(tokenizer.decode(
        expected_original_ids,
        skip_special_tokens=True
    ))

    print("TEACHER-FORCED PREDICTION:")
    print(tokenizer.decode(
        predicted_original_ids,
        skip_special_tokens=True
    ))



TRAINING EXAMPLE 1
EXPECTED:
Discussing the background of Rammohun 's religious crusade, Amitabha Mukherjee hasently observed:' Rammohun was not essentially a religious reformer in the sense that Ramananda, Kabir, Nanak and Chaitanya were during the medieval period of India history.
TEACHER-FORCED PREDICTION:
Theing the  of the Nath,,'s ' de , theaabh e, has ed thed that  akrishna,ohun  aly a a  in  the  of the,, ir  nak, tanya  a the Indiana  of the..

TRAINING EXAMPLE 2
EXPECTED:
shell
TEACHER-FORCED PREDICTION:


TRAINING EXAMPLE 3
EXPECTED:
Which is total not net.
TEACHER-FORCED PREDICTION:
Chich can notly .

TRAINING EXAMPLE 4
EXPECTED:
He noted that a cohesive, responsive and prosperous ASEAN is central to India's Indo-Pacific Vision and contributes to Security And Growth for All in the Region (S).
TEACHER-FORCED PREDICTION:
He said said that a     - to ous ssistance’    ands -  India ( - to in the Securityhra- ( the  India  (.)

TRAINING EXAMPLE 5
EXPECTED:
invest
TEACHER-FORCE

In [56]:

import random

random.seed(42)

sample_indices = random.sample(
    range(len(train_dataset)),
    20
)

for i, idx in enumerate(sample_indices, start=1):
    example = train_dataset[idx]

    source_original = [
        compact_to_original[token_id]
        for token_id in example["source_ids"]
        if token_id not in [PAD_ID, EOS_ID]
    ]

    target_original = [
        compact_to_original[token_id]
        for token_id in example["target_ids"]
        if token_id not in [PAD_ID, EOS_ID]
    ]

    source_text = tokenizer.decode(
        source_original,
        skip_special_tokens=True
    )

    target_text = tokenizer.decode(
        target_original,
        skip_special_tokens=True
    )

    print(f"\n{'='*60}")
    print(f"PAIR {i}")
    print("HINDI:", source_text)
    print("ENGLISH:", target_text)



PAIR 1
HINDI: translate Hindi to English: मेरी यात्रा के दौरान बनी समझ से निःसंदेह हमारे सहयोग को नए स्तर पर पहुंचाने के लिए परस्पर समझ में संवर्धन होगा।
ENGLISH: The understandings that have been reached during my visit will undoubtedly enhance our mutual understanding and take our co - operation to new levels.

PAIR 2
HINDI: translate Hindi to English: कोर्ट डायरी 'के रूप में रखे गये कुछ दस्तावेजों के आधार पर यह साबित किया गया कि 49 यूरोपीयों की हत्या बादशाह की मंजूरी से हुई.
ENGLISH: Some papers described as' court - diary 'were relied upon to prove that 49 Europeans were murdered with the consent of the King.'

PAIR 3
HINDI: translate Hindi to English: गणिका
ENGLISH: prostitute

PAIR 4
HINDI: translate Hindi to English: मौसम दिन-ब-दिन बदलता है।
ENGLISH: The weather varies from day to day.

PAIR 5
HINDI: translate Hindi to English: वहाँ कोई भी न था।
ENGLISH: There was nothing there!

PAIR 6
HINDI: translate Hindi to English: यानादियों का प्यार सरल, प्रत्यक्ष और सहज होता है और प्रेम

In [57]:

import random
import langid

random.seed(42)

sample_indices = random.sample(
    range(len(train_dataset)),
    500
)

english_count = 0
other_count = 0
empty_count = 0

language_counts = {}

for idx in sample_indices:
    example = train_dataset[idx]

    target_original = [
        compact_to_original[token_id]
        for token_id in example["target_ids"]
        if token_id not in [PAD_ID, EOS_ID]
    ]

    target_text = tokenizer.decode(
        target_original,
        skip_special_tokens=True
    ).strip()

    if not target_text:
        empty_count += 1
        continue

    language, confidence = langid.classify(target_text)

    language_counts[language] = (
        language_counts.get(language, 0) + 1
    )

    if language == "en":
        english_count += 1
    else:
        other_count += 1

print("LANGUAGE DISTRIBUTION (500 TRAINING EXAMPLES)")
print("English:", english_count)
print("Other languages:", other_count)
print("Empty:", empty_count)

print("\nDetected languages:")
for language, count in sorted(
    language_counts.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(language, ":", count)


LANGUAGE DISTRIBUTION (500 TRAINING EXAMPLES)
English: 456
Other languages: 42
Empty: 2

Detected languages:
en : 456
de : 6
fr : 6
it : 5
es : 4
pl : 2
lt : 2
id : 2
no : 2
mt : 2
da : 2
cs : 1
tl : 1
nl : 1
fi : 1
ro : 1
et : 1
eu : 1
hr : 1
sv : 1


In [58]:

checkpoint = torch.load(
    r"E:\punctuation_project\models\hindi_english_translation\epoch2_checkpoint.pt",
    map_location="cpu",
    weights_only=False
)

print("CHECKPOINT INFORMATION")

for key in ["epoch", "step", "vocab_size", "d_model"]:
    print(f"{key}: {checkpoint.get(key)}")

print("\nOptimizer learning rate:")

for param_group in checkpoint["optimizer_state_dict"]["param_groups"]:
    print(param_group["lr"])

print("\nSaved model parameter count:")
print(sum(
    parameter.numel()
    for parameter in checkpoint["model_state_dict"].values()
))


CHECKPOINT INFORMATION
epoch: 2
step: 25000
vocab_size: 32000
d_model: 128

Optimizer learning rate:
0.0003

Saved model parameter count:
4791296


In [59]:

from torch.utils.data import DataLoader, Subset

# Use a fixed sample for reproducibility
sample_size = 1000

random.seed(42)
sample_indices = random.sample(
    range(len(train_dataset)),
    sample_size
)

train_sample = Subset(train_dataset, sample_indices)

train_sample_loader = DataLoader(
    train_sample,
    batch_size=8,
    shuffle=False,
    collate_fn=train_loader.collate_fn,
    num_workers=0
)

model.eval()

total_loss = 0
total_tokens = 0

with torch.no_grad():
    for batch in train_sample_loader:
        source_batch = batch["source_ids"].to(device)
        decoder_batch = batch["decoder_ids"].to(device)
        target_batch = batch["target_ids"].to(device)

        logits = model(source_batch, decoder_batch)

        loss_sum = F.cross_entropy(
            logits.reshape(-1, VOCAB_SIZE),
            target_batch.reshape(-1),
            ignore_index=-100,
            reduction="sum"
        )

        total_loss += loss_sum.item()
        total_tokens += (target_batch != -100).sum().item()

print("Training sample examples:", sample_size)
print("Training sample valid tokens:", total_tokens)
print("Training sample loss:", round(total_loss / total_tokens, 4))


Training sample examples: 1000
Training sample valid tokens: 20840
Training sample loss: 3.8821


In [60]:

model.eval()

correct_tokens = 0
total_tokens = 0

with torch.no_grad():
    for batch in train_sample_loader:
        source_batch = batch["source_ids"].to(device)
        decoder_batch = batch["decoder_ids"].to(device)
        target_batch = batch["target_ids"].to(device)

        logits = model(source_batch, decoder_batch)
        predictions = logits.argmax(dim=-1)

        valid_positions = target_batch != -100

        correct_tokens += (
            (predictions == target_batch) & valid_positions
        ).sum().item()

        total_tokens += valid_positions.sum().item()

training_accuracy = (
    100 * correct_tokens / total_tokens
)

print("Training sample token accuracy:",
      round(training_accuracy, 2), "%")

print("Training correct tokens:", correct_tokens)
print("Training total tokens:", total_tokens)


Training sample token accuracy: 33.33 %
Training correct tokens: 6947
Training total tokens: 20840


In [61]:

import shutil
from pathlib import Path

source_checkpoint = Path(
    r"E:\punctuation_project\models\hindi_english_translation\epoch2_checkpoint.pt"
)

backup_checkpoint = Path(
    r"E:\punctuation_project\models\hindi_english_translation\epoch2_baseline_backup.pt"
)

if not backup_checkpoint.exists():
    shutil.copy2(source_checkpoint, backup_checkpoint)

print("Baseline checkpoint preserved:", backup_checkpoint.exists())
print("Backup size (MB):", round(
    backup_checkpoint.stat().st_size / (1024 * 1024), 2
))


Baseline checkpoint preserved: True
Backup size (MB): 54.91


In [62]:

D_MODEL = 256
NUM_HEADS = 4
NUM_ENCODER_LAYERS = 3
NUM_DECODER_LAYERS = 3
FFN_DIM = 512


In [63]:

import gc
import torch
import torch.nn.functional as F

# Release the previous model from GPU memory.
if "model" in globals():
    del model
if "optimizer" in globals():
    del optimizer

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

device = torch.device("cuda")

# Create the larger model.
benchmark_model = HindiEnglishTransformer().to(device)

params = sum(p.numel() for p in benchmark_model.parameters())
print(f"Parameters: {params:,}")
print(f"Model size (FP32): {params * 4 / (1024**2):.2f} MB")

# Use one real batch.
source_batch, decoder_batch, target_batch = next(iter(validation_loader))

source_batch = source_batch.to(device)
decoder_batch = decoder_batch.to(device)
target_batch = target_batch.to(device)

# Measure forward + backward memory.
benchmark_model.train()
logits = benchmark_model(source_batch, decoder_batch)

loss = F.cross_entropy(
    logits.reshape(-1, VOCAB_SIZE),
    target_batch.reshape(-1),
    ignore_index=-100
)

loss.backward()
torch.cuda.synchronize()

print(f"Test loss: {loss.item():.4f}")
print(f"Peak GPU memory: {torch.cuda.max_memory_allocated() / (1024**3):.3f} GB")

# Release benchmark resources.
del benchmark_model, logits, loss
gc.collect()
torch.cuda.empty_cache()


Parameters: 12,211,200
Model size (FP32): 46.58 MB
Test loss: 127.1527
Peak GPU memory: 0.864 GB


In [65]:

import torch

def collate_fn(batch):
    source_sequences = [item["source_ids"] for item in batch]
    target_sequences = [item["target_ids"] for item in batch]

    batch_size = len(batch)
    max_source_len = max(len(x) for x in source_sequences)
    max_target_len = max(len(x) for x in target_sequences)

    source_ids = torch.full(
        (batch_size, max_source_len),
        PAD_ID,
        dtype=torch.long
    )

    decoder_ids = torch.full(
        (batch_size, max_target_len),
        PAD_ID,
        dtype=torch.long
    )

    target_ids = torch.full(
        (batch_size, max_target_len),
        -100,
        dtype=torch.long
    )

    for i, (source, target) in enumerate(
        zip(source_sequences, target_sequences)
    ):
        source_ids[i, :len(source)] = torch.tensor(source)

        decoder_sequence = [START_ID] + target[:-1]

        decoder_ids[i, :len(decoder_sequence)] = torch.tensor(
            decoder_sequence
        )

        target_ids[i, :len(target)] = torch.tensor(target)

    return source_ids, decoder_ids, target_ids

print("Collate function defined successfully.")


Collate function defined successfully.


In [66]:

import gc
import time
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from pathlib import Path

# Configuration
BATCH_SIZE = 8
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0

device = torch.device("cuda")

# Clean up benchmark resources
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Create a fresh model. Do NOT load the old checkpoint.
model = HindiEnglishTransformer().to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

print("Model parameters:", f"{sum(p.numel() for p in model.parameters()):,}")

# Training loader
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=False
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0,
    pin_memory=False
)

# ---------------- TRAINING ----------------
model.train()
total_train_loss = 0.0
train_batches = 0
start_time = time.time()

for step, batch in enumerate(train_loader, start=1):
    source_ids, decoder_ids, target_ids = batch

    source_ids = source_ids.to(device)
    decoder_ids = decoder_ids.to(device)
    target_ids = target_ids.to(device)

    optimizer.zero_grad(set_to_none=True)

    logits = model(source_ids, decoder_ids)

    loss = F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),
        target_ids.reshape(-1),
        ignore_index=-100
    )

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()

    total_train_loss += loss.item()
    train_batches += 1

    if step % 1000 == 0:
        print(
            f"Step {step}/{len(train_loader)} | "
            f"Loss: {loss.item():.4f} | "
            f"Average: {total_train_loss / train_batches:.4f}"
        )

average_train_loss = total_train_loss / train_batches

print("\nTraining completed.")
print("Average training loss:", round(average_train_loss, 4))
print("Training time (minutes):", round((time.time() - start_time) / 60, 2))

# ---------------- VALIDATION ----------------
model.eval()
total_val_loss = 0.0
total_correct = 0
total_tokens = 0

with torch.no_grad():
    for batch in validation_loader:
        source_ids, decoder_ids, target_ids = batch

        source_ids = source_ids.to(device)
        decoder_ids = decoder_ids.to(device)
        target_ids = target_ids.to(device)

        logits = model(source_ids, decoder_ids)

        loss_sum = F.cross_entropy(
            logits.reshape(-1, VOCAB_SIZE),
            target_ids.reshape(-1),
            ignore_index=-100,
            reduction="sum"
        )

        total_val_loss += loss_sum.item()

        predictions = logits.argmax(dim=-1)
        valid_mask = target_ids.ne(-100)

        total_correct += (
            (predictions == target_ids) & valid_mask
        ).sum().item()

        total_tokens += valid_mask.sum().item()

average_val_loss = total_val_loss / total_tokens
val_accuracy = 100 * total_correct / total_tokens

print("\nVALIDATION RESULTS")
print("Validation loss:", round(average_val_loss, 4))
print("Token accuracy:", round(val_accuracy, 2), "%")
print("Valid tokens:", total_tokens)

# ---------------- SAVE CHECKPOINT ----------------
checkpoint_dir = Path(
    r"E:\punctuation_project\models\hindi_english_translation"
)
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = checkpoint_dir / "larger_model_epoch1.pt"

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epoch": 1,
    "step": train_batches,
    "vocab_size": VOCAB_SIZE,
    "d_model": D_MODEL,
    "num_heads": NUM_HEADS,
    "num_encoder_layers": NUM_ENCODER_LAYERS,
    "num_decoder_layers": NUM_DECODER_LAYERS,
    "ffn_dim": FFN_DIM,
    "train_loss": average_train_loss,
    "validation_loss": average_val_loss,
    "validation_accuracy": val_accuracy
}, checkpoint_path)

print("\nCheckpoint saved:", checkpoint_path)
print(
    "Peak GPU memory:",
    round(torch.cuda.max_memory_allocated() / (1024**3), 3),
    "GB"
)


Model parameters: 12,211,200
Step 1000/12500 | Loss: 11.6705 | Average: 16.3318
Step 2000/12500 | Loss: 7.3358 | Average: 12.4568
Step 3000/12500 | Loss: 6.2323 | Average: 10.5967
Step 4000/12500 | Loss: 6.1554 | Average: 9.5287
Step 5000/12500 | Loss: 5.9348 | Average: 8.8527
Step 6000/12500 | Loss: 6.1313 | Average: 8.3776
Step 7000/12500 | Loss: 5.5806 | Average: 8.0245
Step 8000/12500 | Loss: 5.8030 | Average: 7.7462
Step 9000/12500 | Loss: 6.1835 | Average: 7.5214
Step 10000/12500 | Loss: 6.2786 | Average: 7.3345
Step 11000/12500 | Loss: 5.8884 | Average: 7.1765
Step 12000/12500 | Loss: 5.7462 | Average: 7.0392

Training completed.
Average training loss: 6.9762
Training time (minutes): 12.75

VALIDATION RESULTS
Validation loss: 5.32
Token accuracy: 22.78 %
Valid tokens: 103081

Checkpoint saved: E:\punctuation_project\models\hindi_english_translation\larger_model_epoch1.pt
Peak GPU memory: 1.021 GB


In [68]:

# Restore the smaller architecture configuration
D_MODEL = 128
NUM_HEADS = 4
NUM_ENCODER_LAYERS = 2
NUM_DECODER_LAYERS = 2
FFN_DIM = 256

print("Architecture restored:")
print("D_MODEL:", D_MODEL)
print("Encoder layers:", NUM_ENCODER_LAYERS)
print("Decoder layers:", NUM_DECODER_LAYERS)
print("FFN dimension:", FFN_DIM)


Architecture restored:
D_MODEL: 128
Encoder layers: 2
Decoder layers: 2
FFN dimension: 256


In [69]:

import gc
import time
import torch
import torch.nn.functional as F
from pathlib import Path

# Configuration
BATCH_SIZE = 8
device = torch.device("cuda")

checkpoint_dir = Path(
    r"E:\punctuation_project\models\hindi_english_translation"
)

epoch2_path = checkpoint_dir / "epoch2_checkpoint.pt"
epoch3_path = checkpoint_dir / "epoch3_checkpoint.pt"

# Safety: never overwrite an existing epoch-3 checkpoint.
if epoch3_path.exists():
    raise FileExistsError(
        f"{epoch3_path} already exists. Preserve it before proceeding."
    )

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Recreate the SMALLER architecture (128 dimensions).
model = HindiEnglishTransformer().to(device)

# Load epoch-2 weights and optimizer state.
checkpoint = torch.load(
    epoch2_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=0.01
)
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

print("Loaded checkpoint from epoch:", checkpoint["epoch"])
print("Starting training for epoch 3...")

# ---------------- TRAINING ----------------
model.train()
total_train_loss = 0.0
train_batches = 0
start_time = time.time()

for step, batch in enumerate(train_loader, start=1):
    source_ids, decoder_ids, target_ids = batch

    source_ids = source_ids.to(device)
    decoder_ids = decoder_ids.to(device)
    target_ids = target_ids.to(device)

    optimizer.zero_grad(set_to_none=True)

    logits = model(source_ids, decoder_ids)

    loss = F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),
        target_ids.reshape(-1),
        ignore_index=-100
    )

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    total_train_loss += loss.item()
    train_batches += 1

    if step % 1000 == 0:
        print(
            f"Step {step}/{len(train_loader)} | "
            f"Loss: {loss.item():.4f} | "
            f"Average: {total_train_loss / train_batches:.4f}"
        )

average_train_loss = total_train_loss / train_batches

print("\nTraining completed.")
print("Average training loss:", round(average_train_loss, 4))
print("Training time (minutes):", round((time.time() - start_time) / 60, 2))

# ---------------- VALIDATION ----------------
model.eval()
total_val_loss = 0.0
total_correct = 0
total_tokens = 0

with torch.no_grad():
    for batch in validation_loader:
        source_ids, decoder_ids, target_ids = batch

        source_ids = source_ids.to(device)
        decoder_ids = decoder_ids.to(device)
        target_ids = target_ids.to(device)

        logits = model(source_ids, decoder_ids)

        loss_sum = F.cross_entropy(
            logits.reshape(-1, VOCAB_SIZE),
            target_ids.reshape(-1),
            ignore_index=-100,
            reduction="sum"
        )

        total_val_loss += loss_sum.item()

        predictions = logits.argmax(dim=-1)
        valid_mask = target_ids.ne(-100)

        total_correct += (
            (predictions == target_ids) & valid_mask
        ).sum().item()

        total_tokens += valid_mask.sum().item()

average_val_loss = total_val_loss / total_tokens
val_accuracy = 100 * total_correct / total_tokens

print("\nEPOCH 3 VALIDATION")
print("Validation loss:", round(average_val_loss, 4))
print("Token accuracy:", round(val_accuracy, 2), "%")
print("Valid tokens:", total_tokens)

# ---------------- SAVE ----------------
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epoch": 3,
    "step": checkpoint.get("step", 0) + train_batches,
    "vocab_size": VOCAB_SIZE,
    "d_model": D_MODEL,
    "train_loss": average_train_loss,
    "validation_loss": average_val_loss,
    "validation_accuracy": val_accuracy
}, epoch3_path)

print("\nCheckpoint saved:", epoch3_path)
print(
    "Peak GPU memory:",
    round(torch.cuda.max_memory_allocated() / (1024**3), 3),
    "GB"
)


Loaded checkpoint from epoch: 2
Starting training for epoch 3...
Step 1000/12500 | Loss: 3.6408 | Average: 4.0645
Step 2000/12500 | Loss: 4.3335 | Average: 4.0679
Step 3000/12500 | Loss: 3.6933 | Average: 4.0638
Step 4000/12500 | Loss: 3.3208 | Average: 4.0676
Step 5000/12500 | Loss: 4.4011 | Average: 4.0669
Step 6000/12500 | Loss: 3.8210 | Average: 4.0620
Step 7000/12500 | Loss: 4.1310 | Average: 4.0584
Step 8000/12500 | Loss: 4.1612 | Average: 4.0555
Step 9000/12500 | Loss: 4.0830 | Average: 4.0553
Step 10000/12500 | Loss: 3.9136 | Average: 4.0529
Step 11000/12500 | Loss: 4.9029 | Average: 4.0531
Step 12000/12500 | Loss: 4.8943 | Average: 4.0526

Training completed.
Average training loss: 4.0516
Training time (minutes): 9.47

EPOCH 3 VALIDATION
Validation loss: 3.89
Token accuracy: 33.84 %
Valid tokens: 103081

Checkpoint saved: E:\punctuation_project\models\hindi_english_translation\epoch3_checkpoint.pt
Peak GPU memory: 0.94 GB


In [70]:

import torch
import torch.nn.functional as F

model.eval()
device = next(model.parameters()).device

def to_original_ids(compact_ids):
    if isinstance(compact_to_original, dict):
        return [
            compact_to_original.get(int(i), tokenizer.unk_token_id)
            for i in compact_ids
        ]
    return [compact_to_original[int(i)] for i in compact_ids]

def decode_compact(compact_ids):
    original_ids = to_original_ids(compact_ids)
    return tokenizer.decode(
        original_ids,
        skip_special_tokens=True
    )

@torch.no_grad()
def generate_translation(source_ids):
    source = torch.tensor(
        source_ids, dtype=torch.long, device=device
    ).unsqueeze(0)

    source_padding_mask = source.eq(PAD_ID)

    source_positions = torch.arange(
        source.size(1), device=device
    ).unsqueeze(0)

    source_embeddings = (
        model.embedding(source)
        + model.source_position(source_positions)
    )

    memory = model.encoder(
        model.dropout(source_embeddings),
        src_key_padding_mask=source_padding_mask
    )

    generated = [START_ID]

    for _ in range(MAX_TARGET_LENGTH):
        decoder_ids = torch.tensor(
            generated, dtype=torch.long, device=device
        ).unsqueeze(0)

        target_length = decoder_ids.size(1)

        target_positions = torch.arange(
            target_length, device=device
        ).unsqueeze(0)

        target_embeddings = (
            model.embedding(decoder_ids)
            + model.target_position(target_positions)
        )

        target_padding_mask = decoder_ids.eq(PAD_ID).clone()
        target_padding_mask[:, 0] = False

        causal_mask = torch.triu(
            torch.ones(
                target_length,
                target_length,
                dtype=torch.bool,
                device=device
            ),
            diagonal=1
        )

        decoded = model.decoder(
            tgt=model.dropout(target_embeddings),
            memory=memory,
            tgt_mask=causal_mask,
            tgt_key_padding_mask=target_padding_mask,
            memory_key_padding_mask=source_padding_mask
        )

        logits = F.linear(decoded, model.embedding.weight)
        next_token = logits[:, -1, :].argmax(dim=-1).item()

        if next_token == EOS_ID:
            break

        generated.append(next_token)

    return generated[1:]


# Test five validation examples
for i in range(5):
    example = validation_dataset[i]

    predicted_ids = generate_translation(example["source_ids"])

    print(f"\nEXAMPLE {i + 1}")
    print("EXPECTED:", decode_compact(example["target_ids"]))
    print("PREDICTED:", decode_compact(predicted_ids))



EXAMPLE 1
EXPECTED: In their concluding speeches Maulana 'Abul Kalam Azad, the Congress president, and Gandhiji made it clear that their next step would be to approach the Viceroy, as representing the British Government, and to appeal to the heads of the principal United Nations for an honourable settlement, which while recognizing the freedom of India, would also advance the cause of the United Nations in the struggle against the aggressor Axis powers.
PREDICTED: The Congress 's called the Congress and the Congress Gandhiji would be given to the British Governments in the British Governments, the British Government British Governments will be their their rights of the British Governments, British Governments, which they will be their their rights of the British Governments of the British Governments of the British Governments, British Governments, British Governments, their British Governments, their British British Governments, British Governments, their British

EXAMPLE 2
EXPECTED:

In [71]:

import gc
import time
import torch
import torch.nn.functional as F
from pathlib import Path

device = torch.device("cuda")

checkpoint_dir = Path(
    r"E:\punctuation_project\models\hindi_english_translation"
)

epoch3_path = checkpoint_dir / "epoch3_checkpoint.pt"
epoch4_path = checkpoint_dir / "epoch4_checkpoint.pt"

if epoch4_path.exists():
    raise FileExistsError("Epoch 4 checkpoint already exists.")

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Load the smaller architecture
model = HindiEnglishTransformer().to(device)

checkpoint = torch.load(
    epoch3_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.01
)

optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

# Lower learning rate
for group in optimizer.param_groups:
    group["lr"] = 1e-4

print("Loaded epoch:", checkpoint["epoch"])
print("Learning rate:", optimizer.param_groups[0]["lr"])

# ---------------- TRAINING ----------------
model.train()
total_train_loss = 0.0
start_time = time.time()

for step, batch in enumerate(train_loader, start=1):
    source_ids, decoder_ids, target_ids = [
        x.to(device) for x in batch
    ]

    optimizer.zero_grad(set_to_none=True)

    logits = model(source_ids, decoder_ids)

    loss = F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),
        target_ids.reshape(-1),
        ignore_index=-100
    )

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    total_train_loss += loss.item()

    if step % 1000 == 0:
        print(
            f"Step {step}/{len(train_loader)} | "
            f"Loss: {loss.item():.4f} | "
            f"Average: {total_train_loss / step:.4f}"
        )

average_train_loss = total_train_loss / len(train_loader)

# ---------------- VALIDATION ----------------
model.eval()
total_loss = 0.0
correct = 0
total_tokens = 0

with torch.no_grad():
    for batch in validation_loader:
        source_ids, decoder_ids, target_ids = [
            x.to(device) for x in batch
        ]

        logits = model(source_ids, decoder_ids)

        total_loss += F.cross_entropy(
            logits.reshape(-1, VOCAB_SIZE),
            target_ids.reshape(-1),
            ignore_index=-100,
            reduction="sum"
        ).item()

        mask = target_ids.ne(-100)
        correct += (
            (logits.argmax(dim=-1) == target_ids) & mask
        ).sum().item()

        total_tokens += mask.sum().item()

val_loss = total_loss / total_tokens
val_accuracy = 100 * correct / total_tokens

# ---------------- SAVE ----------------
torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epoch": 4,
    "step": checkpoint.get("step", 0) + len(train_loader),
    "vocab_size": VOCAB_SIZE,
    "d_model": D_MODEL,
    "train_loss": average_train_loss,
    "validation_loss": val_loss,
    "validation_accuracy": val_accuracy
}, epoch4_path)

print("\nEPOCH 4 RESULTS")
print("Training loss:", round(average_train_loss, 4))
print("Validation loss:", round(val_loss, 4))
print("Token accuracy:", round(val_accuracy, 2), "%")
print("Peak GPU memory:", round(
    torch.cuda.max_memory_allocated() / (1024**3), 3
), "GB")
print("Saved:", epoch4_path)


Loaded epoch: 3
Learning rate: 0.0001
Step 1000/12500 | Loss: 3.2912 | Average: 3.7917
Step 2000/12500 | Loss: 4.2403 | Average: 3.7700
Step 3000/12500 | Loss: 3.9143 | Average: 3.7600
Step 4000/12500 | Loss: 4.1890 | Average: 3.7578
Step 5000/12500 | Loss: 4.0892 | Average: 3.7548
Step 6000/12500 | Loss: 4.1113 | Average: 3.7551
Step 7000/12500 | Loss: 3.6866 | Average: 3.7511
Step 8000/12500 | Loss: 3.5868 | Average: 3.7486
Step 9000/12500 | Loss: 3.5335 | Average: 3.7461
Step 10000/12500 | Loss: 3.4272 | Average: 3.7474
Step 11000/12500 | Loss: 3.7199 | Average: 3.7477
Step 12000/12500 | Loss: 3.7289 | Average: 3.7479

EPOCH 4 RESULTS
Training loss: 3.7478
Validation loss: 3.7382
Token accuracy: 35.83 %
Peak GPU memory: 0.939 GB
Saved: E:\punctuation_project\models\hindi_english_translation\epoch4_checkpoint.pt


In [72]:

import torch
import torch.nn.functional as F

checkpoint = torch.load(
    r"E:\punctuation_project\models\hindi_english_translation\epoch4_checkpoint.pt",
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Epoch 4 checkpoint loaded.")

for i in range(5):
    example = validation_dataset[i]

    predicted_ids = generate_translation(example["source_ids"])

    print(f"\nEXAMPLE {i + 1}")
    print("EXPECTED:", decode_compact(example["target_ids"]))
    print("PREDICTED:", decode_compact(predicted_ids))


Epoch 4 checkpoint loaded.

EXAMPLE 1
EXPECTED: In their concluding speeches Maulana 'Abul Kalam Azad, the Congress president, and Gandhiji made it clear that their next step would be to approach the Viceroy, as representing the British Government, and to appeal to the heads of the principal United Nations for an honourable settlement, which while recognizing the freedom of India, would also advance the cause of the United Nations in the struggle against the aggressor Axis powers.
PREDICTED: The Congress would be called the Gandhiji and Gandhiji would have taken a British Government in the British Government, which they will be a British Government of the British Governments of the British Governments of the British Governments, the British Government would have their rights and their rights of the rights of the their rights of the their rights of the British rights of the parties.

EXAMPLE 2
EXPECTED: The revolutionary prisoners in Bengal and 'where who were not included in the genera

In [73]:

from datasets import load_from_disk
from pathlib import Path

original_path = Path(
    r"E:\punctuation_project\data\processed\translation_processed"
)

original_data = load_from_disk(str(original_path))

print("ORIGINAL DATASET")
print(original_data)

print("\nTRAINING EXAMPLES:", len(original_data["train"]))
print("VALIDATION EXAMPLES:", len(original_data["validation"]))
print("TEST EXAMPLES:", len(original_data["test"]))

print("\nTRAINING COLUMNS:", original_data["train"].column_names)

print("\nFIRST TRAINING EXAMPLE:")
print("Input IDs:", original_data["train"][0]["input_ids"][:20])
print("Target IDs:", original_data["train"][0]["labels"][:20])


ORIGINAL DATASET
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1320349
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 165044
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 165044
    })
})

TRAINING EXAMPLES: 1320349
VALIDATION EXAMPLES: 165044
TEST EXAMPLES: 165044

TRAINING COLUMNS: ['input_ids', 'attention_mask', 'labels']

FIRST TRAINING EXAMPLE:
Input IDs: [37194, 8857, 288, 5413, 267, 1890, 8133, 2312, 4324, 1162, 1100, 1484, 1896, 58936, 996, 259, 6783, 1]
Target IDs: [5398, 1272, 337, 380, 746, 63324, 70545, 363, 121839, 27969, 1]


In [74]:
import shutil
from pathlib import Path

model_dir = Path(
    r"E:\punctuation_project\models\hindi_english_translation"
)

source = model_dir / "epoch4_checkpoint.pt"
backup = model_dir / "epoch4_baseline_backup.pt"

if not source.exists():
    raise FileNotFoundError(f"Checkpoint not found: {source}")

if backup.exists():
    raise FileExistsError(
        f"Backup already exists: {backup}. Not overwriting it."
    )

shutil.copy2(source, backup)

print("BASELINE BACKUP SAVED")
print("File:", backup)
print("Size:", round(backup.stat().st_size / (1024 * 1024), 2), "MB")
print("Backup verified:", backup.exists())
print("Original checkpoint preserved:", source.exists())

BASELINE BACKUP SAVED
File: E:\punctuation_project\models\hindi_english_translation\epoch4_baseline_backup.pt
Size: 54.91 MB
Backup verified: True
Original checkpoint preserved: True


In [76]:
import json
from pathlib import Path

mapping_path = Path(
    r"E:\punctuation_project\models\hindi_english_translation\vocabulary_mapping.json"
)

with open(mapping_path, "r", encoding="utf-8") as f:
    mapping_data = json.load(f)

print("Type:", type(mapping_data))
print("Length:", len(mapping_data))

print("\nFirst 10 entries:")
print(mapping_data[:10] if isinstance(mapping_data, list)
      else list(mapping_data.items())[:10])

Type: <class 'list'>
Length: 32000

First 10 entries:
[0, 1, 2, 259, 288, 267, 8857, 5413, 37194, 261]


In [77]:
import json
from pathlib import Path

mapping_path = Path(
    r"E:\punctuation_project\models\hindi_english_translation\vocabulary_mapping.json"
)

with open(mapping_path, "r", encoding="utf-8") as f:
    selected_token_ids = json.load(f)

# Original token ID -> Compact token ID
original_to_compact = {
    original_id: compact_id
    for compact_id, original_id in enumerate(selected_token_ids)
}

# Compact token ID -> Original token ID
compact_to_original = {
    compact_id: original_id
    for compact_id, original_id in enumerate(selected_token_ids)
}

print("Compact vocabulary size:", len(selected_token_ids))
print("Original-to-compact mapping:", len(original_to_compact))
print("Compact-to-original mapping:", len(compact_to_original))

print("\nMapping verification:")
print("Original ID 259 -> Compact ID:", original_to_compact[259])
print("Compact ID 3 -> Original ID:", compact_to_original[3])

Compact vocabulary size: 32000
Original-to-compact mapping: 32000
Compact-to-original mapping: 32000

Mapping verification:
Original ID 259 -> Compact ID: 3
Compact ID 3 -> Original ID: 259


In [78]:

import random
from pathlib import Path
from datasets import DatasetDict

output_path = Path(
    r"E:\punctuation_project\data\processed\translation_compact_300k"
)

# Reproducible random selection
random.seed(42)

train_indices = random.sample(
    range(len(original_data["train"])), 300_000
)

validation_indices = random.sample(
    range(len(original_data["validation"])), 5_000
)

train_subset = original_data["train"].select(train_indices)
validation_subset = original_data["validation"].select(validation_indices)

# Convert original IDs into compact vocabulary IDs
def convert_batch(batch):
    converted_sources = []
    converted_targets = []

    for source, attention, target in zip(
        batch["input_ids"],
        batch["attention_mask"],
        batch["labels"]
    ):
        source = [
            token for token, mask in zip(source, attention)
            if mask == 1
        ]

        target = [token for token in target if token >= 0]

        compact_source = [
            original_to_compact.get(token, 2)
            for token in source
        ]

        compact_target = [
            original_to_compact.get(token, 2)
            for token in target
        ]

        converted_sources.append(compact_source)
        converted_targets.append(compact_target)

    return {
        "source_ids": converted_sources,
        "target_ids": converted_targets
    }

train_compact = train_subset.map(
    convert_batch,
    batched=True,
    batch_size=1000,
    remove_columns=train_subset.column_names,
    desc="Converting training data"
)

validation_compact = validation_subset.map(
    convert_batch,
    batched=True,
    batch_size=1000,
    remove_columns=validation_subset.column_names,
    desc="Converting validation data"
)

compact_dataset = DatasetDict({
    "train": train_compact,
    "validation": validation_compact
})

compact_dataset.save_to_disk(str(output_path))

print("\nDATASET PREPARATION COMPLETE")
print("Training:", len(compact_dataset["train"]))
print("Validation:", len(compact_dataset["validation"]))
print("Saved:", output_path)


Converting training data:   0%|          | 0/300000 [00:00<?, ? examples/s]

Converting validation data:   0%|          | 0/5000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/300000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]


DATASET PREPARATION COMPLETE
Training: 300000
Validation: 5000
Saved: E:\punctuation_project\data\processed\translation_compact_300k


In [79]:
import torch
from pathlib import Path
from datasets import load_from_disk
from torch.utils.data import DataLoader

# Keep the exact epoch-4 architecture
VOCAB_SIZE = 32000
D_MODEL = 128
NUM_HEADS = 4
NUM_ENCODER_LAYERS = 2
NUM_DECODER_LAYERS = 2
FFN_DIM = 256

BATCH_SIZE = 8
LEARNING_RATE = 1e-4

model_dir = Path(
    r"E:\punctuation_project\models\hindi_english_translation"
)

# 1. Load expanded dataset
dataset_path = Path(
    r"E:\punctuation_project\data\processed\translation_compact_300k"
)

expanded_dataset = load_from_disk(str(dataset_path))

train_dataset = expanded_dataset["train"]
validation_dataset = expanded_dataset["validation"]

# 2. Restore epoch-4 checkpoint
checkpoint_path = model_dir / "epoch4_checkpoint.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

model = HindiEnglishTransformer().to(device)
model.load_state_dict(checkpoint["model_state_dict"])

# 3. Restore optimizer state
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01
)

optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

# Explicitly set the learning rate
for param_group in optimizer.param_groups:
    param_group["lr"] = LEARNING_RATE

# 4. Prepare data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=0
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=0
)

print("MODEL RESTORED")
print("Checkpoint epoch:", checkpoint["epoch"])
print("Checkpoint step:", checkpoint["step"])
print("Model parameters:", sum(p.numel() for p in model.parameters()))
print("Training batches:", len(train_loader))
print("Validation batches:", len(validation_loader))
print("GPU:", device)

MODEL RESTORED
Checkpoint epoch: 4
Checkpoint step: 50000
Model parameters: 4791296
Training batches: 37500
Validation batches: 625
GPU: cuda


In [80]:

import torch
import torch.nn.functional as F
import time
from pathlib import Path

model.train()

total_loss = 0.0
start_time = time.time()

starting_step = 50000
checkpoint_interval = 5000

torch.cuda.reset_peak_memory_stats()

for step, batch in enumerate(train_loader, start=1):

    source_ids, decoder_ids, target_ids = batch

    source_ids = source_ids.to(device)
    decoder_ids = decoder_ids.to(device)
    target_ids = target_ids.to(device)

    optimizer.zero_grad(set_to_none=True)

    logits = model(source_ids, decoder_ids)

    loss = F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),
        target_ids.reshape(-1),
        ignore_index=-100
    )

    loss.backward()
    optimizer.step()

    total_loss += loss.item()

    if step % 1000 == 0:
        print(
            f"Step {step}/{len(train_loader)} | "
            f"Loss: {loss.item():.4f}"
        )

    # Save recovery checkpoints every 5000 steps
    if step % checkpoint_interval == 0:
        checkpoint_data = {
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "epoch": 5,
            "step": starting_step + step,
            "vocab_size": VOCAB_SIZE,
            "d_model": D_MODEL
        }

        checkpoint_file = (
            model_dir / f"epoch5_step_{starting_step + step}.pt"
        )

        torch.save(checkpoint_data, checkpoint_file)

        print("Checkpoint saved:", checkpoint_file.name)

average_train_loss = total_loss / len(train_loader)

training_time = (time.time() - start_time) / 60

# Save final epoch-5 checkpoint
final_checkpoint = {
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "epoch": 5,
    "step": starting_step + len(train_loader),
    "vocab_size": VOCAB_SIZE,
    "d_model": D_MODEL
}

final_path = model_dir / "epoch5_checkpoint.pt"
torch.save(final_checkpoint, final_path)

peak_memory = torch.cuda.max_memory_allocated() / (1024 ** 3)

print("\nEPOCH 5 TRAINING COMPLETE")
print("Average training loss:", round(average_train_loss, 4))
print("Training time:", round(training_time, 2), "minutes")
print("Peak GPU memory:", round(peak_memory, 3), "GB")
print("Final checkpoint:", final_path)


Step 1000/37500 | Loss: 3.4960
Step 2000/37500 | Loss: 3.9997
Step 3000/37500 | Loss: 4.5539
Step 4000/37500 | Loss: 3.5946
Step 5000/37500 | Loss: 3.9548
Checkpoint saved: epoch5_step_55000.pt
Step 6000/37500 | Loss: 3.8700
Step 7000/37500 | Loss: 4.0796
Step 8000/37500 | Loss: 3.5074
Step 9000/37500 | Loss: 3.4071
Step 10000/37500 | Loss: 3.2581
Checkpoint saved: epoch5_step_60000.pt
Step 11000/37500 | Loss: 3.8129
Step 12000/37500 | Loss: 4.6181
Step 13000/37500 | Loss: 3.4386
Step 14000/37500 | Loss: 3.7084
Step 15000/37500 | Loss: 3.6796
Checkpoint saved: epoch5_step_65000.pt
Step 16000/37500 | Loss: 3.8269
Step 17000/37500 | Loss: 4.0037
Step 18000/37500 | Loss: 4.1338
Step 19000/37500 | Loss: 2.8178
Step 20000/37500 | Loss: 3.6808
Checkpoint saved: epoch5_step_70000.pt
Step 21000/37500 | Loss: 4.5620
Step 22000/37500 | Loss: 4.4454
Step 23000/37500 | Loss: 3.4089
Step 24000/37500 | Loss: 3.0444
Step 25000/37500 | Loss: 4.6530
Checkpoint saved: epoch5_step_75000.pt
Step 26000/375

In [81]:
import torch
import torch.nn.functional as F

model.eval()

total_loss = 0.0
total_correct = 0
total_tokens = 0

with torch.no_grad():
    for batch in validation_loader:

        source_ids, decoder_ids, target_ids = batch

        source_ids = source_ids.to(device)
        decoder_ids = decoder_ids.to(device)
        target_ids = target_ids.to(device)

        logits = model(source_ids, decoder_ids)

        loss_sum = F.cross_entropy(
            logits.reshape(-1, VOCAB_SIZE),
            target_ids.reshape(-1),
            ignore_index=-100,
            reduction="sum"
        )

        predictions = logits.argmax(dim=-1)
        valid_mask = target_ids.ne(-100)

        total_loss += loss_sum.item()
        total_correct += (
            (predictions == target_ids) & valid_mask
        ).sum().item()

        total_tokens += valid_mask.sum().item()

validation_loss = total_loss / total_tokens
token_accuracy = 100 * total_correct / total_tokens

print("EPOCH 5 VALIDATION RESULTS")
print("Validation examples:", len(validation_dataset))
print("Valid tokens:", total_tokens)
print("Validation loss:", round(validation_loss, 4))
print("Token accuracy:", round(token_accuracy, 2), "%")

EPOCH 5 VALIDATION RESULTS
Validation examples: 5000
Valid tokens: 104890
Validation loss: 3.5755
Token accuracy: 37.14 %


In [82]:
model.eval()

for i in range(5):
    example = validation_dataset[i]

    predicted_ids = generate_translation(
        example["source_ids"]
    )

    expected = decode_compact(example["target_ids"])
    predicted = decode_compact(predicted_ids)

    print(f"\nEXAMPLE {i+1}")
    print("EXPECTED :", expected)
    print("PREDICTED:", predicted)


EXAMPLE 1
EXPECTED : It is remarkable for the construction of a silken - thread - lined cabin of sticks, bits of leaves, chewed plant fibres, sand grains, small pebbles of selected sizes and mosaic of colours, even small molluscan shells, etc.
PREDICTED: This 's small - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -

EXAMPLE 2
EXPECTED : Saka
PREDICTED: 

EXAMPLE 3
EXPECTED : Could not open the containing folder
PREDICTED: Could not open folder

EXAMPLE 4
EXPECTED : In ancient time and in Middle Age the “”Shaivs ( of Lord ) “”, “”Shaktas ( of deity of power) “” and Vaishnava ( of Lord Vishnu) all were fighting, to them many saints of that era attempted successfully in getting accord and told the Mutual interdependence of all sects.
PREDICTED: In the ancient ancient - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 

EXAMPLE 5
EXPECTED : An unre

In [83]:
import shutil
from pathlib import Path

model_dir = Path(
    r"E:\punctuation_project\models\hindi_english_translation"
)

source = model_dir / "epoch5_checkpoint.pt"
backup = model_dir / "epoch5_final_backup.pt"

if not source.exists():
    raise FileNotFoundError(source)

if backup.exists():
    raise FileExistsError(
        "Final backup already exists. Not overwriting it."
    )

shutil.copy2(source, backup)

print("FINAL MODEL BACKUP SAVED")
print("Location:", backup)
print("Size:", round(backup.stat().st_size / (1024 * 1024), 2), "MB")
print("Verified:", backup.exists())

FINAL MODEL BACKUP SAVED
Location: E:\punctuation_project\models\hindi_english_translation\epoch5_final_backup.pt
Size: 54.91 MB
Verified: True


In [1]:

import json
from pathlib import Path

PROJECT_DIR = Path(r"E:\punctuation_project")

# 1. Inspect model directories
print("========== MODEL DIRECTORIES ==========")

models_dir = PROJECT_DIR / "models"

for path in sorted(models_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_DIR))

# 2. Inspect translation notebook
print("\n========== TRANSLATION NOTEBOOK ==========")

notebook_path = PROJECT_DIR / "03_translation.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

for i, cell in enumerate(notebook["cells"]):

    source = "".join(cell["source"]).strip()

    if source:
        print(f"\n--- Cell {i} ({cell['cell_type']}) ---")
        print(source[:1800])

# 3. Inspect translation datasets
print("\n========== TRANSLATION DATASETS ==========")

from datasets import load_from_disk

for name in [
    "translation_compact_32k",
    "translation_compact_300k",
    "translation_processed"
]:
    path = PROJECT_DIR / "data" / "processed" / name

    if path.exists():
        dataset = load_from_disk(str(path))

        print(f"\n{name}")

        for split in dataset:
            print(split, len(dataset[split]))
            print("Columns:", dataset[split].column_names)

        print("Sample:", dataset["train"][0])


========== MODEL DIRECTORIES ==========
models\hindi_english_translation\epoch2_baseline_backup.pt
models\hindi_english_translation\epoch2_checkpoint.pt
models\hindi_english_translation\epoch3_checkpoint.pt
models\hindi_english_translation\epoch4_baseline_backup.pt
models\hindi_english_translation\epoch4_checkpoint.pt
models\hindi_english_translation\epoch5_checkpoint.pt
models\hindi_english_translation\epoch5_final_backup.pt
models\hindi_english_translation\epoch5_step_55000.pt
models\hindi_english_translation\epoch5_step_60000.pt
models\hindi_english_translation\epoch5_step_65000.pt
models\hindi_english_translation\epoch5_step_70000.pt
models\hindi_english_translation\epoch5_step_75000.pt
models\hindi_english_translation\epoch5_step_80000.pt
models\hindi_english_translation\epoch5_step_85000.pt
models\hindi_english_translation\larger_model_epoch1.pt
models\hindi_english_translation\last_checkpoint.pt
models\hindi_english_translation\vocabulary_mapping.json
models\indicbert_punctuatio

In [2]:

from pathlib import Path
import torch

MODEL_DIR = Path(
    r"E:\punctuation_project\models\hindi_english_translation"
)

print("TRANSLATION CHECKPOINT INVENTORY")
print("=" * 60)

checkpoint_files = sorted(MODEL_DIR.glob("*.pt"))

for path in checkpoint_files:
    print(f"\n{path.name}")
    print(f"Size: {path.stat().st_size / (1024 ** 2):.2f} MB")

    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    print("Epoch:", checkpoint.get("epoch", "Not recorded"))
    print("Step:", checkpoint.get("step", "Not recorded"))

    state = checkpoint.get("model_state_dict")

    if state:
        print("Embedding shape:",
              tuple(state["embedding.weight"].shape))
        print("Parameter count:",
              sum(t.numel() for t in state.values()))

    del checkpoint

print("\nInventory complete.")


TRANSLATION CHECKPOINT INVENTORY

epoch2_baseline_backup.pt
Size: 54.91 MB
Epoch: 2
Step: 25000
Embedding shape: (32000, 128)
Parameter count: 4791296

epoch2_checkpoint.pt
Size: 54.91 MB
Epoch: 2
Step: 25000
Embedding shape: (32000, 128)
Parameter count: 4791296

epoch3_checkpoint.pt
Size: 54.91 MB
Epoch: 3
Step: 37500
Embedding shape: (32000, 128)
Parameter count: 4791296

epoch4_baseline_backup.pt
Size: 54.91 MB
Epoch: 4
Step: 50000
Embedding shape: (32000, 128)
Parameter count: 4791296

epoch4_checkpoint.pt
Size: 54.91 MB
Epoch: 4
Step: 50000
Embedding shape: (32000, 128)
Parameter count: 4791296

epoch5_checkpoint.pt
Size: 54.91 MB
Epoch: 5
Step: 87500
Embedding shape: (32000, 128)
Parameter count: 4791296

epoch5_final_backup.pt
Size: 54.91 MB
Epoch: 5
Step: 87500
Embedding shape: (32000, 128)
Parameter count: 4791296

epoch5_step_55000.pt
Size: 54.91 MB
Epoch: 5
Step: 55000
Embedding shape: (32000, 128)
Parameter count: 4791296

epoch5_step_60000.pt
Size: 54.91 MB
Epoch: 5
Step:

In [3]:

from pathlib import Path
import torch

MODEL_DIR = Path(
    r"E:\punctuation_project\models\hindi_english_translation"
)

print(f"{'CHECKPOINT':<35} {'EPOCH':<8} {'STEP':<12} {'SIZE (MB)':<10}")
print("-" * 70)

for path in sorted(MODEL_DIR.glob("*.pt")):
    checkpoint = torch.load(
        path,
        map_location="cpu",
        weights_only=False
    )

    epoch = checkpoint.get("epoch", "-")
    step = checkpoint.get("step", "-")
    size = path.stat().st_size / (1024 ** 2)

    print(f"{path.name:<35} {str(epoch):<8} {str(step):<12} {size:<10.2f}")

    del checkpoint


CHECKPOINT                          EPOCH    STEP         SIZE (MB) 
----------------------------------------------------------------------
epoch2_baseline_backup.pt           2        25000        54.91     
epoch2_checkpoint.pt                2        25000        54.91     
epoch3_checkpoint.pt                3        37500        54.91     
epoch4_baseline_backup.pt           4        50000        54.91     
epoch4_checkpoint.pt                4        50000        54.91     
epoch5_checkpoint.pt                5        87500        54.91     
epoch5_final_backup.pt              5        87500        54.91     
epoch5_step_55000.pt                5        55000        54.91     
epoch5_step_60000.pt                5        60000        54.91     
epoch5_step_65000.pt                5        65000        54.91     
epoch5_step_70000.pt                5        70000        54.91     
epoch5_step_75000.pt                5        75000        54.91     
epoch5_step_80000.pt            